# SI7016 — Procesamiento de Lenguaje Natural Aplicado
## Proyecto Final — Entrenador Personal Inteligente (Gimnasio)

**Universidad EAFIT · Maestría en Ciencias de Datos y Analítica · 2026-1**


Alvaro Javier Mutis Guerrero

Jose Luis Bedoya Martinez

---

### Descripción general

Este proyecto implementa un **Sistema Multiagente Inteligente** para el acompañamiento de clientes de un gimnasio, integrando:

- **LLM GPT-4** (OpenAI API) para razonamiento y generación de lenguaje natural.
- **MCP (Model Context Protocol)** como protocolo estándar de comunicación entre agentes.
- **RAG (Retrieval Augmented Generation)** sobre documentos reales de entrenamiento, nutrición y rendimiento almacenados en Google Drive.
- **Embeddings multilingües** para la recuperación semántica de contexto.
- **Fine-tuning ligero** sobre un corpus de Q&A del dominio fitness.
- **Interfaz conversacional** (Gradio) con entrada por texto y por voz.
- **Memoria episódica** por cliente (lifelong-learning simulado).

---

### Agentes del sistema

| Agente | Rol |
|---|---|
| **Entrenador** | Licenciado en Educación Física. Diseña rutinas y progresiones. |
| **Nutricionista** | Profesional de nutrición deportiva. Calcula TMB, TDEE y macros. |
| **Analista de Rendimiento** | Registra medidas iniciales, establece objetivos, hace seguimientos. |
| **Cliente (4 perfiles)** | Mujer 25 post-lipo · Hombre 29 hipertrofia · Hombre 55 sobrepeso con rodilla · Mujer 18 subir músculo. |
| **Coordinador** | Orquesta los agentes según la intención del usuario. |

---

### Componente de innovación (futuro del NLP)

- **NLP multimodal**: entrada por voz (Whisper) + salida textual enriquecida.
- **Comunicación inter-agente** bajo el estándar emergente **MCP**.
- **Aprendizaje continuo simulado**: memoria persistente por cliente que se resume y se reinyecta en cada sesión.
- **Combinación de datos estructurados** (JSON: perfiles, ejercicios, macros) **y no estructurados** (PDFs, notas del gimnasio) en un solo pipeline.

In [1]:
# ========================================================================
#  Instalación de dependencias del proyecto
# ========================================================================
# Instalamos todas las librerías necesarias:
#   - openai           -> cliente oficial de la API de GPT-4
#   - mcp              -> Model Context Protocol (comunicación entre agentes)
#   - langchain + chroma -> orquestación de RAG y vector store
#   - sentence-transformers -> embeddings multilingües (HuggingFace)
#   - gradio           -> interfaz conversacional web
#   - openai-whisper   -> speech-to-text para la entrada por voz
#   - pypdf, docx2txt  -> lectura de documentos de la base RAG
# ========================================================================

%%capture
!pip install -q "openai>=1.40.0"
!pip install -q "mcp[cli]>=1.27.0" python-dotenv nest_asyncio
!pip install -q langchain langchain-community langchain-openai langchain-huggingface
!pip install -q langchain-chroma chromadb
!pip install -q sentence-transformers tiktoken
!pip install -q pypdf docx2txt unstructured[md]
!pip install -q gradio
!pip install -q openai-whisper
!pip install -q pandas scikit-learn matplotlib

In [2]:
# ========================================================================
#  Verificación: comprobamos que cada librería clave importa sin errores
# ========================================================================
import importlib

paquetes = [
    "openai",
    "mcp",
    "langchain",
    "langchain_community",
    "langchain_chroma",
    "chromadb",
    "sentence_transformers",
    "gradio",
    "whisper",
    "pypdf",
]

print("Verificación de dependencias")
print("-" * 40)
for p in paquetes:
    try:
        importlib.import_module(p)
        print(f"  OK    {p}")
    except Exception as e:
        print(f"  FALLA {p}  ->  {e}")

Verificación de dependencias
----------------------------------------
  OK    openai
  OK    mcp
  OK    langchain
  OK    langchain_community
  OK    langchain_chroma
  OK    chromadb
  OK    sentence_transformers
  OK    gradio
  OK    whisper
  OK    pypdf


# 1. Acceso a la base documental en Google Drive

El sistema RAG consulta documentos reales del gimnasio almacenados en tu Drive en la carpeta:

**`Mi unidad / rag_agente_entrenador /`**

con la siguiente estructura:

```
rag_agente_entrenador/
├── RENDIMIENTO/        # documentos sobre análisis de rendimiento y seguimiento
├── NUTRICION/          # material de nutrición deportiva
├── ENTRENADOR/         # protocolos, técnica de ejercicios, periodización
└── CLIENTES/
    ├── POST_CIRUGIA/   # protocolos post-lipo y recuperación
    ├── OBESIDAD/       # manejo de clientes con sobrepeso
    ├── HIPERTROFIA/    # ganancia de masa muscular
    └── BAJAR/          # pérdida de peso
```

En los siguientes pasos:

1. Montaremos Google Drive en Colab.
2. Verificaremos que todas las subcarpetas existan.
3. Listaremos los archivos disponibles por categoría.

In [3]:
# ========================================================================
#  Montar Google Drive en Colab
# ========================================================================
# Esto abrirá una ventana emergente para autorizar el acceso.
# Tras la autorización, tu Drive quedará disponible en /content/drive/
# ========================================================================

from google.colab import drive
drive.mount('/content/drive')

print("Google Drive montado correctamente en /content/drive")

Mounted at /content/drive
Google Drive montado correctamente en /content/drive


In [4]:
# ========================================================================
#  Detectar la carpeta rag_agente_entrenador
# ========================================================================
# Ajusta la variable DRIVE_DOCS si tu carpeta está en otra ubicación.
# ========================================================================

from pathlib import Path

DRIVE_DOCS = Path('/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/rag_agente_entrenador')

assert DRIVE_DOCS.exists(), (
    f"No se encontró la carpeta {DRIVE_DOCS}.\n"
    "Revisa que el nombre y la ubicación en Drive sean correctos."
)

print("Carpeta base encontrada:", DRIVE_DOCS)
print("-" * 60)

# Verificamos las 4 subcarpetas principales
subcarpetas_esperadas = ['RENDIMIENTO', 'NUTRICION', 'ENTRENADOR', 'CLIENTES']
for sub in subcarpetas_esperadas:
    p = DRIVE_DOCS / sub
    estado = "OK   " if p.exists() else "FALTA"
    print(f"  {estado}  {p.name}")

# Verificamos las subcarpetas dentro de CLIENTES
print("-" * 60)
subclientes = ['POST_CIRUGIA', 'OBESIDAD', 'HIPERTROFIA', 'BAJAR']
for sub in subclientes:
    p = DRIVE_DOCS / 'CLIENTES' / sub
    estado = "OK   " if p.exists() else "FALTA"
    print(f"  {estado}  CLIENTES/{sub}")

Carpeta base encontrada: /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/rag_agente_entrenador
------------------------------------------------------------
  OK     RENDIMIENTO
  OK     NUTRICION
  OK     ENTRENADOR
  OK     CLIENTES
------------------------------------------------------------
  OK     CLIENTES/POST_CIRUGIA
  OK     CLIENTES/OBESIDAD
  OK     CLIENTES/HIPERTROFIA
  OK     CLIENTES/BAJAR


In [5]:
# ========================================================================
#  Inventario de documentos disponibles
# ========================================================================
# Recorremos recursivamente todas las subcarpetas y listamos los archivos
# con extensión válida para RAG (.pdf, .docx, .txt, .md).
# ========================================================================

extensiones_validas = {'.pdf', '.docx', '.doc', '.txt', '.md'}

# Agrupamos por subcarpeta de primer nivel
from collections import defaultdict
inventario = defaultdict(list)

for archivo in DRIVE_DOCS.rglob('*'):
    if archivo.is_file() and archivo.suffix.lower() in extensiones_validas:
        ruta_relativa = archivo.relative_to(DRIVE_DOCS)
        categoria_raiz = ruta_relativa.parts[0]
        inventario[categoria_raiz].append(str(ruta_relativa))

print("INVENTARIO DE LA BASE DOCUMENTAL")
print("=" * 60)
total = 0
for categoria, archivos in sorted(inventario.items()):
    print(f"\n[{categoria}]  ({len(archivos)} archivos)")
    for a in archivos[:15]:   # mostramos máximo 15 por categoría para no saturar
        print(f"   - {a}")
    if len(archivos) > 15:
        print(f"   ... y {len(archivos) - 15} más")
    total += len(archivos)

print("\n" + "=" * 60)
print(f"TOTAL de documentos detectados: {total}")

INVENTARIO DE LA BASE DOCUMENTAL

[CLIENTES]  (16 archivos)
   - CLIENTES/HIPERTROFIA/Nutrition for Sport, Exercise and Performance.pdf
   - CLIENTES/HIPERTROFIA/Physical_Activity_Guidelines_2nd_edition.pdf
   - CLIENTES/HIPERTROFIA/programming-and-periodization.pdf
   - CLIENTES/HIPERTROFIA/8weekmassbuildinghypertrophyworkout.pdf
   - CLIENTES/HIPERTROFIA/rev02_raya.pdf
   - CLIENTES/HIPERTROFIA/growing_stronger.pdf
   - CLIENTES/HIPERTROFIA/Nutrition-Guide.pdf
   - CLIENTES/HIPERTROFIA/OS2016-High-Performance-Nutrition-Program.pdf
   - CLIENTES/HIPERTROFIA/AEDN_2022_nutricion_deportiva_borrador_V0.pdf
   - CLIENTES/POST_CIRUGIA/hip-exercises-print-guide.pdf
   - CLIENTES/POST_CIRUGIA/Liposuction-Post-op-Instructions.pdf
   - CLIENTES/OBESIDAD/ob_gdlns.pdf
   - CLIENTES/OBESIDAD/Exercise for Older Adults.pdf
   - CLIENTES/OBESIDAD/growing_stronger.pdf
   - CLIENTES/BAJAR/AEDN_2022_nutricion_deportiva_borrador_V0.pdf
   ... y 1 más

[ENTRENADOR]  (13 archivos)
   - ENTRENADOR/Physical_

# 2. Configuración del modelo de lenguaje (GPT-4)

Todos los agentes del sistema (Entrenador, Nutricionista, Analista, Coordinador) usarán un único modelo central: **GPT-4** a través de la API oficial de OpenAI.

## Modelos configurados

| Constante | Valor | Uso |
|---|---|---|
| `LLM_MODEL` | `gpt-4o` | Razonamiento principal de los agentes y tool-calling |
| `LLM_MODEL_FAST` | `gpt-4o-mini` | Tareas auxiliares (coordinador, resúmenes de memoria) |
| `EMBED_MODEL` | `intfloat/multilingual-e5-small` | Embeddings locales multilingües para RAG |

Usar dos variantes de GPT-4 permite:

- Controlar **costos** (el coordinador hace rutas simples, no necesita el modelo grande).
- Mantener **calidad alta** cuando el agente realmente razona y llama herramientas.
- Poder **cambiar fácilmente** a `gpt-4-turbo` u otros si se quisiera.

## Seguridad de la API key

La clave se pide interactivamente con `getpass`, que **no la imprime** en el notebook ni queda guardada en el archivo. Se almacena solo en la variable de entorno del runtime de Colab.

In [6]:
# ========================================================================
#  Cargar la API key de OpenAI de forma segura
# ========================================================================
# La clave se pide por pantalla y se guarda solo en la variable de entorno
# del runtime de Colab (no queda escrita en el notebook).
# ========================================================================

import os
from getpass import getpass

if 'OPENAI_API_KEY' not in os.environ or not os.environ['OPENAI_API_KEY']:
    os.environ['OPENAI_API_KEY'] = getpass('Ingresa tu OPENAI_API_KEY: ')

# Validación mínima del formato
key = os.environ['OPENAI_API_KEY']
assert key.startswith('sk-'), "La API key no parece válida (debe empezar con 'sk-')."
print(f"API key cargada correctamente (longitud: {len(key)} caracteres)")

Ingresa tu OPENAI_API_KEY: ··········
API key cargada correctamente (longitud: 164 caracteres)


In [7]:
# ========================================================================
#  Constantes globales del sistema
# ========================================================================
# Definimos aquí los modelos para poder cambiarlos en un solo lugar.
# ========================================================================

LLM_MODEL       = "gpt-4o"                           # razonamiento principal
LLM_MODEL_FAST  = "gpt-4o-mini"                      # coordinador y tareas auxiliares
EMBED_MODEL     = "intfloat/multilingual-e5-small"   # embeddings locales gratis

print("Configuración de modelos:")
print(f"  LLM principal   : {LLM_MODEL}")
print(f"  LLM auxiliar    : {LLM_MODEL_FAST}")
print(f"  Embeddings RAG  : {EMBED_MODEL}")

Configuración de modelos:
  LLM principal   : gpt-4o
  LLM auxiliar    : gpt-4o-mini
  Embeddings RAG  : intfloat/multilingual-e5-small


In [8]:
# ========================================================================
#  Prueba de conectividad con GPT-4
# ========================================================================


from openai import OpenAI
import time

oai = OpenAI()

t0 = time.time()
respuesta = oai.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system",
         "content": "Eres un entrenador personal profesional. Responde en una frase."},
        {"role": "user",
         "content": "¿Qué es la hipertrofia muscular?"}
    ],
    temperature=0.3,
    max_tokens=100,
)
t1 = time.time()

print("Respuesta de", LLM_MODEL)
print("-" * 60)
print(respuesta.choices[0].message.content)
print("-" * 60)
print(f"Latencia: {t1 - t0:.2f} s")
print(f"Tokens usados: {respuesta.usage.total_tokens}")

Respuesta de gpt-4o
------------------------------------------------------------
La hipertrofia muscular es el aumento del tamaño de las fibras musculares debido al entrenamiento de resistencia y una adecuada nutrición.
------------------------------------------------------------
Latencia: 2.73 s
Tokens usados: 58


In [9]:
# ========================================================================
#  Prueba del modelo auxiliar (gpt-4o-mini)
# ========================================================================


t0 = time.time()
respuesta = oai.chat.completions.create(
    model=LLM_MODEL_FAST,
    messages=[
        {"role": "system",
         "content": "Clasifica si la pregunta es de ENTRENAMIENTO, NUTRICION o RENDIMIENTO. Responde solo con la palabra."},
        {"role": "user",
         "content": "¿Cuántos gramos de proteína debo comer al día?"}
    ],
    temperature=0,
    max_tokens=10,
)
t1 = time.time()

print("Clasificación :", respuesta.choices[0].message.content.strip())
print(f"Latencia      : {t1 - t0:.2f} s")
print(f"Tokens        : {respuesta.usage.total_tokens}")

Clasificación : NUTRICION
Latencia      : 1.75 s
Tokens        : 53


# 3. Estructura del proyecto

Creamos la siguiente jerarquía de carpetas en el runtime de Colab:

```
proyecto_entrenador/
├── data/         # JSONs estructurados: perfiles, base de ejercicios, macros
├── docs/         # Documentos sincronizados desde Google Drive (fuente del RAG)
│   ├── RENDIMIENTO/
│   ├── NUTRICION/
│   ├── ENTRENADOR/
│   └── CLIENTES/
│       ├── POST_CIRUGIA/
│       ├── OBESIDAD/
│       ├── HIPERTROFIA/
│       └── BAJAR/
├── src/          # Código del servidor MCP y de los agentes
├── rag_store/    # Vector store Chroma (embeddings persistidos)
├── memory/       # Memoria episódica por cliente (JSONL)
└── outputs/      # Transcripciones y resultados de ejecución
```

## ¿Por qué copiar los documentos?

- **Velocidad**: lectura local es 10-50x más rápida que leer desde el mount de Drive.
- **Reproducibilidad**: el pipeline RAG trabaja siempre sobre los mismos archivos.
- **Metadatos**: al copiar preservamos la carpeta padre (RENDIMIENTO, NUTRICION, etc.) como **etiqueta de categoría** que usaremos para filtrar la recuperación.

In [10]:
# ========================================================================
#  Crear la estructura base del proyecto
# ========================================================================

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador')

subcarpetas = [
    'data',
    'src',
    'docs',
    'rag_store',
    'memory',
    'outputs',
]

for s in subcarpetas:
    (PROJECT_DIR / s).mkdir(parents=True, exist_ok=True)

print("Estructura creada en:", PROJECT_DIR)
print("-" * 60)
for s in sorted(PROJECT_DIR.iterdir()):
    print(f"  {s.name}/")

Estructura creada en: /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador
------------------------------------------------------------
  data/
  docs/
  memory/
  outputs/
  rag_store/
  src/


In [11]:
# ========================================================================
#  PASO 5.2 - Sincronizar documentos desde Drive a docs/ local
# ========================================================================
# Copiamos solo archivos con extensiones válidas para RAG y preservamos
# la estructura de subcarpetas (para usarlas como metadato 'categoria').
# ========================================================================

import shutil
from collections import defaultdict

SRC = DRIVE_DOCS                       # viene del Paso 3
DST = PROJECT_DIR / 'docs'

extensiones_validas = {'.pdf', '.docx', '.doc', '.txt', '.md'}
copiados = defaultdict(list)

for origen in SRC.rglob('*'):
    if origen.is_file() and origen.suffix.lower() in extensiones_validas:
        rel = origen.relative_to(SRC)
        destino = DST / rel
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(origen, destino)

        categoria = rel.parts[0]       # primera subcarpeta = categoría
        copiados[categoria].append(str(rel))

print("SINCRONIZACIÓN COMPLETADA")
print("=" * 60)
total = 0
for cat, files in sorted(copiados.items()):
    print(f"  [{cat:15s}]  {len(files)} archivos copiados")
    total += len(files)

print("-" * 60)
print(f"TOTAL copiado a {DST}: {total} archivos")

SINCRONIZACIÓN COMPLETADA
  [CLIENTES       ]  16 archivos copiados
  [ENTRENADOR     ]  13 archivos copiados
  [NUTRICION      ]  10 archivos copiados
  [RENDIMIENTO    ]  3 archivos copiados
------------------------------------------------------------
TOTAL copiado a /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/docs: 42 archivos


In [12]:
# ========================================================================
#   Verificación: árbol de carpetas y tamaño total
# ========================================================================

def arbol(path: Path, prefijo: str = "", max_archivos: int = 3):
    """Imprime un árbol compacto de carpetas y archivos."""
    entradas = sorted(path.iterdir(), key=lambda p: (not p.is_dir(), p.name))
    dirs = [e for e in entradas if e.is_dir()]
    files = [e for e in entradas if e.is_file()]

    for d in dirs:
        print(f"{prefijo}├── {d.name}/")
        arbol(d, prefijo + "│   ", max_archivos)

    for f in files[:max_archivos]:
        print(f"{prefijo}├── {f.name}")
    if len(files) > max_archivos:
        print(f"{prefijo}└── ... y {len(files) - max_archivos} archivo(s) más")

print(f"{PROJECT_DIR}/")
arbol(PROJECT_DIR / 'docs', prefijo="")

# Tamaño total en MB
tam_bytes = sum(f.stat().st_size for f in (PROJECT_DIR / 'docs').rglob('*') if f.is_file())
print("\nTamaño total de la base documental local: {:.2f} MB".format(tam_bytes / 1024 / 1024))

/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/
├── CLIENTES/
│   ├── BAJAR/
│   │   ├── AEDN_2022_nutricion_deportiva_borrador_V0.pdf
│   │   ├── OS2016-High-Performance-Nutrition-Program.pdf
│   ├── HIPERTROFIA/
│   │   ├── 8weekmassbuildinghypertrophyworkout.pdf
│   │   ├── AEDN_2022_nutricion_deportiva_borrador_V0.pdf
│   │   ├── Nutrition for Sport, Exercise and Performance.pdf
│   │   └── ... y 6 archivo(s) más
│   ├── OBESIDAD/
│   │   ├── Exercise for Older Adults.pdf
│   │   ├── growing_stronger.pdf
│   │   ├── ob_gdlns.pdf
│   ├── POST_CIRUGIA/
│   │   ├── Liposuction-Post-op-Instructions.pdf
│   │   ├── hip-exercises-print-guide.pdf
├── ENTRENADOR/
│   ├── 8weekmassbuildinghypertrophyworkout.pdf
│   ├── Are the Hypertrophic Adaptations to High and LowLoad Resistance Training Muscle Fiber Type.pdf
│   ├── Loading Recommendations for Muscle Strength, Hypertrophy, and Local Endurance A Re-Examination of the Repetition Continuum.pdf
│   └─

# 4. Datos estructurados del sistema

Creamos cuatro archivos JSON en `data/` que serán consultados por el servidor MCP:

| Archivo | Contenido | Uso |
|---|---|---|
| `gym_context.json` | Info del gimnasio, temas cubiertos, filosofía | Prompt del sistema, rutas de enrutamiento |
| `client_profiles.json` | Los 4 clientes de la definicion de agentes Clientes | Personalización de cada agente |
| `exercise_db.json` | Catálogo de ejercicios con restricciones | Recomendaciones seguras |
| `nutrition_db.json` | Factores kcal, macros por objetivo | Cálculos nutricionales |

## Los 4 perfiles de clientes (del enunciado)

| ID | Nombre | Edad | Sexo | Estatura | Peso | % Grasa | Objetivo | Restricciones |
|---|---|---|---|---|---|---|---|---|
| **C1** | Mariana | 25 | F | 165 cm | 58 kg | 22 % | Mantener medidas post-lipo | Sin impacto 6 sem |
| **C2** | Andrés  | 29 | M | 180 cm | 82 kg | 14.9 % | Hipertrofia + definir | — |
| **C3** | Jorge   | 55 | M | 172 cm | 95 kg | 32 % | Bajar peso | Sin impacto en rodillas |
| **C4** | Valentina | 18 | F | 175 cm | 50 kg | 11 % | Ganar músculo | — |

> Nota: los datos marcados como "sin información" en el enunciado (Jorge sin datos confirmados, Valentina sin info nutricional) se guardan como placeholders y el agente **Analista de Rendimiento** los pedirá en la primera conversación.

## ¿Por qué separar estructurado y no estructurado?

- El LLM es bueno explicando conceptos → **RAG sobre los PDFs**.
- El LLM es malo calculando → **tools MCP sobre JSON estructurados**.
- Al separarlos reducimos alucinaciones y ganamos trazabilidad.

In [13]:
# ========================================================================
#  Archivo: gym_context.json
# ========================================================================
# Define quién es el gimnasio, su filosofía, los temas que maneja y
# palabras clave por área (útiles para el Coordinador).
# ========================================================================

import json

gym_context = {
    "gym_name": "FitCoach AI",
    "motto": "Entrenamiento personalizado respaldado por datos",
    "specialties": [
        "hipertrofia",
        "perdida de peso",
        "post-cirugia",
        "rehabilitacion",
        "acondicionamiento general"
    ],
    "topics": {
        "entrenamiento": {
            "name": "Entrenamiento y acondicionamiento fisico",
            "keywords": ["rutina", "fuerza", "hipertrofia", "cardio", "HIIT",
                         "series", "repeticiones", "progresion", "periodizacion"],
            "core_idea": "La progresion de cargas y la periodizacion son clave para resultados sostenibles."
        },
        "nutricion": {
            "name": "Nutricion deportiva",
            "keywords": ["macronutrientes", "proteina", "carbohidratos",
                         "grasas", "deficit", "superavit", "calorias", "tdee", "tmb"],
            "core_idea": "El deficit o superavit calorico define la direccion; los macros definen la calidad."
        },
        "rendimiento": {
            "name": "Analisis de rendimiento y seguimiento",
            "keywords": ["peso", "imc", "grasa", "perimetros", "seguimiento",
                         "progreso", "medidas", "antropometria"],
            "core_idea": "Medir semanalmente peso, % grasa y fuerza permite ajustar el plan antes de estancarse."
        },
        "post_cirugia": {
            "name": "Rehabilitacion post-cirugia",
            "keywords": ["lipo", "liposuccion", "recuperacion", "cuidados",
                         "compresion", "postoperatorio"],
            "core_idea": "Ejercicio progresivo y de baja carga mecanica durante las primeras 6-8 semanas."
        }
    }
}

ruta = PROJECT_DIR / 'data' / 'gym_context.json'
ruta.write_text(json.dumps(gym_context, ensure_ascii=False, indent=2), encoding='utf-8')

print(f"Creado: {ruta.name}")
print(f"Temas definidos: {list(gym_context['topics'].keys())}")

Creado: gym_context.json
Temas definidos: ['entrenamiento', 'nutricion', 'rendimiento', 'post_cirugia']


In [14]:
# ========================================================================
#  Archivo: client_profiles.json
# ========================================================================
# Los 4 clientes del enunciado. Los campos con 'sin_info' seran llenados
# en la primera conversacion por el Analista de Rendimiento.
# ========================================================================

client_profiles = [
    {
        "id": "C1",
        "name": "Mariana",
        "gender": "F",
        "age": 25,
        "height_cm": 165,
        "weight_kg": 58,
        "body_fat_pct": 22.0,
        "category": "POST_CIRUGIA",
        "goal": "Mantener medidas obtenidas tras lipo-escultura",
        "consistency": "media",
        "discipline": "media",
        "weekly_frequency": 3,
        "restrictions": [
            "sin impacto las primeras 6 semanas",
            "evitar abdominales con carga hasta alta 4 semanas",
            "uso de prenda de compresion"
        ],
        "notes": "Post-operatorio de lipo-escultura en recuperacion. Datos iniciales estimados."
    },
    {
        "id": "C2",
        "name": "Andres",
        "gender": "M",
        "age": 29,
        "height_cm": 180,
        "weight_kg": 82,
        "body_fat_pct": 14.9,
        "category": "HIPERTROFIA",
        "goal": "Aumentar masa muscular y lograr mayor definicion",
        "consistency": "alta",
        "discipline": "baja",
        "weekly_frequency": 5,
        "restrictions": [],
        "notes": "Trabaja en oficina, entrena de noche. Disciplina baja requiere plan simple y automatizado."
    },
    {
        "id": "C3",
        "name": "Jorge",
        "gender": "M",
        "age": 55,
        "height_cm": 172,
        "weight_kg": 95,
        "body_fat_pct": 32.0,
        "category": "BAJAR",
        "goal": "Bajar de peso y mejorar condicion general",
        "consistency": "sin_info",
        "discipline": "sin_info",
        "weekly_frequency": None,
        "restrictions": [
            "sin impacto en rodillas",
            "inicio luego de vida sedentaria",
            "recomendacion medica previa"
        ],
        "notes": "Llega por recomendacion medica. Peso, altura y % grasa deben confirmarse en la primera cita."
    },
    {
        "id": "C4",
        "name": "Valentina",
        "gender": "F",
        "age": 18,
        "height_cm": 175,
        "weight_kg": 50,
        "body_fat_pct": 11.0,
        "category": "HIPERTROFIA",
        "goal": "Aumentar volumen muscular con progreso demostrable",
        "consistency": "alta",
        "discipline": "alta",
        "weekly_frequency": 5,
        "restrictions": [],
        "notes": "Bajo peso y bajo musculo. Alimentacion 'sin info': requiere plan nutricional con superavit guiado."
    }
]

ruta = PROJECT_DIR / 'data' / 'client_profiles.json'
ruta.write_text(json.dumps(client_profiles, ensure_ascii=False, indent=2), encoding='utf-8')

print(f"Creado: {ruta.name} con {len(client_profiles)} clientes")
for c in client_profiles:
    print(f"  [{c['id']}] {c['name']:10s} - {c['category']:12s} - objetivo: {c['goal']}")

Creado: client_profiles.json con 4 clientes
  [C1] Mariana    - POST_CIRUGIA - objetivo: Mantener medidas obtenidas tras lipo-escultura
  [C2] Andres     - HIPERTROFIA  - objetivo: Aumentar masa muscular y lograr mayor definicion
  [C3] Jorge      - BAJAR        - objetivo: Bajar de peso y mejorar condicion general
  [C4] Valentina  - HIPERTROFIA  - objetivo: Aumentar volumen muscular con progreso demostrable


In [15]:
# ========================================================================
#  Archivo: nutrition_db.json
# ========================================================================
# Parametros nutricionales por objetivo. Usado por el agente Nutricionista
# para calcular kcal y distribucion de macros.
# ========================================================================

nutrition_db = {
    "proteinas_g_por_kg": {
        "mantenimiento": 1.2,
        "hipertrofia":   1.8,
        "perdida":       2.0,
        "post_cirugia":  1.5
    },
    "kcal_ajuste_pct": {
        "mantenimiento":  0,
        "hipertrofia":   10,    # +10% sobre TDEE
        "perdida":      -20,    # -20% sobre TDEE
        "post_cirugia":   0
    },
    "reparto_macros": {
        "hipertrofia":   {"prot": 0.30, "carb": 0.45, "gras": 0.25},
        "perdida":       {"prot": 0.35, "carb": 0.35, "gras": 0.30},
        "mantenimiento": {"prot": 0.25, "carb": 0.45, "gras": 0.30},
        "post_cirugia":  {"prot": 0.30, "carb": 0.40, "gras": 0.30}
    },
    "hidratacion_ml_por_kg": 35,
    "comidas_por_dia_recomendadas": {
        "hipertrofia": 4,
        "perdida":     3,
        "mantenimiento": 3,
        "post_cirugia": 4
    }
}

ruta = PROJECT_DIR / 'data' / 'nutrition_db.json'
ruta.write_text(json.dumps(nutrition_db, ensure_ascii=False, indent=2), encoding='utf-8')

print(f"Creado: {ruta.name}")
print("  Objetivos cubiertos:", list(nutrition_db['reparto_macros'].keys()))

Creado: nutrition_db.json
  Objetivos cubiertos: ['hipertrofia', 'perdida', 'mantenimiento', 'post_cirugia']


In [16]:
# ========================================================================
#  Verificacion de los 4 JSONs creados
# ========================================================================

data_dir = PROJECT_DIR / 'data'
print("Archivos en data/:")
print("-" * 60)
for f in sorted(data_dir.iterdir()):
    if f.suffix == '.json':
        contenido = json.loads(f.read_text(encoding='utf-8'))
        n = len(contenido) if isinstance(contenido, list) else len(contenido.keys())
        tam_kb = f.stat().st_size / 1024
        print(f"  {f.name:28s}  {n} entradas top-level  ({tam_kb:.1f} KB)")

# Cargamos los clientes en memoria para el resto del notebook
CLIENT_MAP = {c['id']: c for c in json.loads((data_dir / 'client_profiles.json').read_text('utf-8'))}
print("\nCLIENT_MAP en memoria:", list(CLIENT_MAP.keys()))

Archivos en data/:
------------------------------------------------------------
  client_profiles.json          4 entradas top-level  (2.0 KB)
  exercise_db.json              17 entradas top-level  (2.3 KB)
  gym_context.json              4 entradas top-level  (1.7 KB)
  nutrition_db.json             5 entradas top-level  (0.8 KB)

CLIENT_MAP en memoria: ['C1', 'C2', 'C3', 'C4']


# 5. Servidor MCP del gimnasio

El servidor `mcp_server_gym.py` centraliza:

## Resources (datos de solo lectura)

| URI | Contenido |
|---|---|
| `gym://overview` | Información general del gimnasio (nombre, filosofía, temas) |
| `client://{client_id}` | Perfil completo de un cliente (C1..C4) |
| `exercises://catalog` | Catálogo de ejercicios con restricciones |

## Prompts (plantillas de sistema)

| Nombre | Propósito |
|---|---|
| `adaptive_trainer` | Prompt personalizado del Entrenador según el cliente |
| `nutrition_coach` | Prompt del Nutricionista con datos del cliente ya inyectados |

## Tools (funciones ejecutables)

| Tool | Descripción |
|---|---|
| `get_client(client_id)` | Devuelve el perfil estructurado del cliente |
| `compute_imc(weight_kg, height_cm)` | IMC + categoría OMS |
| `compute_tmb_tdee(weight, height, age, gender, activity)` | Fórmula Mifflin-St Jeor + factor de actividad |
| `plan_macros(tdee, goal)` | kcal objetivo + gramos de proteína, carbos y grasas |
| `recommend_exercises(category, restricciones)` | Filtra el catálogo excluyendo contraindicaciones |
| `register_measurement(client_id, weight, fat, note)` | Añade una entrada al histórico |
| `list_progress(client_id)` | Devuelve el histórico de medidas |
| `detect_risk(client_id)` | Detecta estancamiento o pérdida demasiado rápida |

## Comunicación

El servidor se ejecuta como subproceso y habla por **STDIO** con el cliente (JSON-RPC). Esta es la arquitectura canónica de MCP y permite reemplazar fácilmente el servidor local por uno remoto (HTTP) si se quisiera.

In [17]:


import json
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador')
data_dir = PROJECT_DIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

# -------- 1) gym_context.json --------
gym_context = {
    "gym_name": "FitCoach AI",
    "motto": "Entrenamiento personalizado respaldado por datos",
    "specialties": ["hipertrofia","perdida de peso","post-cirugia","rehabilitacion","acondicionamiento general"],
    "topics": {
        "entrenamiento": {"name":"Entrenamiento y acondicionamiento fisico",
                          "keywords":["rutina","fuerza","hipertrofia","cardio","HIIT","series","repeticiones","progresion","periodizacion"],
                          "core_idea":"La progresion de cargas y la periodizacion son clave para resultados sostenibles."},
        "nutricion": {"name":"Nutricion deportiva",
                      "keywords":["macronutrientes","proteina","carbohidratos","grasas","deficit","superavit","calorias","tdee","tmb"],
                      "core_idea":"El deficit o superavit calorico define la direccion; los macros definen la calidad."},
        "rendimiento": {"name":"Analisis de rendimiento y seguimiento",
                        "keywords":["peso","imc","grasa","perimetros","seguimiento","progreso","medidas","antropometria"],
                        "core_idea":"Medir semanalmente peso, % grasa y fuerza permite ajustar el plan antes de estancarse."},
        "post_cirugia": {"name":"Rehabilitacion post-cirugia",
                         "keywords":["lipo","liposuccion","recuperacion","cuidados","compresion","postoperatorio"],
                         "core_idea":"Ejercicio progresivo y de baja carga mecanica durante las primeras 6-8 semanas."}
    }
}

# -------- 2) client_profiles.json --------
client_profiles = [
    {"id":"C1","name":"Mariana","gender":"F","age":25,"height_cm":165,"weight_kg":58,"body_fat_pct":22.0,
     "category":"POST_CIRUGIA","goal":"Mantener medidas obtenidas tras lipo-escultura",
     "consistency":"media","discipline":"media","weekly_frequency":3,
     "restrictions":["sin impacto las primeras 6 semanas","evitar abdominales con carga hasta la 4a semana","uso de prenda de compresion"],
     "notes":"Post-operatorio de lipo-escultura en recuperacion. Datos iniciales estimados."},
    {"id":"C2","name":"Andres","gender":"M","age":29,"height_cm":180,"weight_kg":82,"body_fat_pct":14.9,
     "category":"HIPERTROFIA","goal":"Aumentar masa muscular y lograr mayor definicion",
     "consistency":"alta","discipline":"baja","weekly_frequency":5,"restrictions":[],
     "notes":"Trabaja en oficina, entrena de noche. Plan simple y automatizado."},
    {"id":"C3","name":"Jorge","gender":"M","age":55,"height_cm":172,"weight_kg":95,"body_fat_pct":32.0,
     "category":"BAJAR","goal":"Bajar de peso y mejorar condicion general",
     "consistency":"sin_info","discipline":"sin_info","weekly_frequency":None,
     "restrictions":["sin impacto en rodillas","inicio luego de vida sedentaria","recomendacion medica previa"],
     "notes":"Llega por recomendacion medica. Peso, altura y % grasa deben confirmarse en la primera cita."},
    {"id":"C4","name":"Valentina","gender":"F","age":18,"height_cm":175,"weight_kg":50,"body_fat_pct":11.0,
     "category":"HIPERTROFIA","goal":"Aumentar volumen muscular con progreso demostrable",
     "consistency":"alta","discipline":"alta","weekly_frequency":5,"restrictions":[],
     "notes":"Bajo peso y bajo musculo. Plan nutricional con superavit guiado."}
]

# -------- 3) exercise_db.json --------
exercise_db = {
    "sentadilla_goblet":         {"grupo":"piernas","nivel":"intermedio","impacto":"bajo","contraindicado":["rodilla_grave"]},
    "prensa_45":                 {"grupo":"piernas","nivel":"principiante","impacto":"bajo","contraindicado":[]},
    "hip_thrust":                {"grupo":"gluteos","nivel":"intermedio","impacto":"bajo","contraindicado":[]},
    "peso_muerto":               {"grupo":"espalda","nivel":"avanzado","impacto":"medio","contraindicado":["lumbar","post_cirugia"]},
    "remo_polea_baja":           {"grupo":"espalda","nivel":"principiante","impacto":"bajo","contraindicado":[]},
    "press_banca":               {"grupo":"pecho","nivel":"intermedio","impacto":"bajo","contraindicado":["hombro_agudo"]},
    "press_militar":             {"grupo":"hombro","nivel":"intermedio","impacto":"bajo","contraindicado":["hombro_agudo"]},
    "curl_biceps_mancuernas":    {"grupo":"brazos","nivel":"principiante","impacto":"bajo","contraindicado":[]},
    "extension_triceps_polea":   {"grupo":"brazos","nivel":"principiante","impacto":"bajo","contraindicado":[]},
    "plancha":                   {"grupo":"core","nivel":"principiante","impacto":"bajo","contraindicado":["post_cirugia_abdominal_1-4sem"]},
    "eliptica":                  {"grupo":"cardio","nivel":"principiante","impacto":"bajo","contraindicado":[]},
    "bicicleta_estatica":        {"grupo":"cardio","nivel":"principiante","impacto":"bajo","contraindicado":[]},
    "caminadora_inclinada":      {"grupo":"cardio","nivel":"principiante","impacto":"medio","contraindicado":[]},
    "burpee":                    {"grupo":"full_body","nivel":"avanzado","impacto":"alto","contraindicado":["rodilla","post_cirugia"]},
    "saltos_caja":               {"grupo":"piernas","nivel":"avanzado","impacto":"alto","contraindicado":["rodilla","post_cirugia"]},
    "natacion_suave":            {"grupo":"full_body","nivel":"principiante","impacto":"nulo","contraindicado":[]},
    "movilidad_articular":       {"grupo":"mobility","nivel":"principiante","impacto":"nulo","contraindicado":[]},
}

# -------- 4) nutrition_db.json --------
nutrition_db = {
    "proteinas_g_por_kg": {"mantenimiento":1.2,"hipertrofia":1.8,"perdida":2.0,"post_cirugia":1.5},
    "kcal_ajuste_pct":    {"mantenimiento":0,"hipertrofia":10,"perdida":-20,"post_cirugia":0},
    "reparto_macros": {
        "hipertrofia":   {"prot":0.30,"carb":0.45,"gras":0.25},
        "perdida":       {"prot":0.35,"carb":0.35,"gras":0.30},
        "mantenimiento": {"prot":0.25,"carb":0.45,"gras":0.30},
        "post_cirugia":  {"prot":0.30,"carb":0.40,"gras":0.30},
    },
    "hidratacion_ml_por_kg": 35,
    "comidas_por_dia_recomendadas": {"hipertrofia":4,"perdida":3,"mantenimiento":3,"post_cirugia":4},
}

# -------- Escritura de los 4 archivos --------
archivos = {
    "gym_context.json":      gym_context,
    "client_profiles.json":  client_profiles,
    "exercise_db.json":      exercise_db,
    "nutrition_db.json":     nutrition_db,
}

print("Regenerando archivos en", data_dir)
print("-" * 60)
for nombre, obj in archivos.items():
    p = data_dir / nombre
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    n = len(obj) if isinstance(obj, list) else len(obj.keys())
    print(f"  OK   {nombre:25s}  ({n} entradas, {p.stat().st_size} bytes)")

# Recargamos en memoria por si se usa después
CLIENT_MAP = {c["id"]: c for c in client_profiles}
print("\nCLIENT_MAP en memoria:", list(CLIENT_MAP.keys()))

Regenerando archivos en /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/data
------------------------------------------------------------
  OK   gym_context.json           (4 entradas, 1740 bytes)
  OK   client_profiles.json       (4 entradas, 2021 bytes)
  OK   exercise_db.json           (17 entradas, 2400 bytes)
  OK   nutrition_db.json          (5 entradas, 769 bytes)

CLIENT_MAP en memoria: ['C1', 'C2', 'C3', 'C4']


In [18]:
# ========================================================================
#  DIAGNOSTICO - ¿por que muere el servidor MCP?
# ========================================================================

import subprocess, sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador')

# ------------------------------------------------------------------------
# Test 1: ¿el modulo del servidor se importa sin errores?
# ------------------------------------------------------------------------
print(">>> TEST 1: importación del módulo del servidor")
p = subprocess.run(
    [sys.executable, '-c',
     'import sys; sys.path.insert(0, r"{}"); '
     'import mcp_server_gym; print("IMPORT OK"); print(mcp_server_gym.mcp)'
     .format(str(PROJECT_DIR / 'src'))],
    capture_output=True, text=True, timeout=20
)
print("return code:", p.returncode)
print("STDOUT    :", p.stdout)
print("STDERR    :", p.stderr)

# ------------------------------------------------------------------------
# Test 2: arrancar el servidor como subproceso con stdin vacio

# ------------------------------------------------------------------------
print("\n>>> TEST 2: ejecutar el servidor como subproceso")
try:
    p = subprocess.run(
        [sys.executable, str(PROJECT_DIR / 'src' / 'mcp_server_gym.py')],
        input='',
        capture_output=True, text=True, timeout=5
    )
    print("return code:", p.returncode)
    print("STDOUT    :", p.stdout[:800])
    print("STDERR    :", p.stderr[:800])
except subprocess.TimeoutExpired as e:
    print("TIMEOUT (ESTO ES BUENO: el server arrancó y quedó escuchando).")
    stderr_parcial = e.stderr.decode() if e.stderr else "(sin stderr)"
    print("STDERR parcial:")
    print(stderr_parcial[:800])

>>> TEST 1: importación del módulo del servidor
return code: 0
STDOUT    : IMPORT OK

STDERR    : 

>>> TEST 2: ejecutar el servidor como subproceso
return code: 0
STDOUT    : 
STDERR    : 


In [19]:
# ========================================================================
#  HELPER MCP - version robusta
#  Define:
#    - open_mcp_session()  -> context manager para abrir una sesion MCP
#    - mcp_result(resp)    -> extrae el resultado de una call_tool()
#    - run_async(coro)     -> ejecuta y desempaca ExceptionGroup
# ========================================================================

import os, sys, json, asyncio, nest_asyncio, traceback
from contextlib import asynccontextmanager
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()
SERVER = str(PROJECT_DIR / 'src' / 'mcp_server_gym.py')


@asynccontextmanager
async def open_mcp_session():
    """Sesion MCP con stderr aislado. Abre y cierra devnull en cada uso."""
    errlog = open(os.devnull, 'w')
    params = StdioServerParameters(command=sys.executable, args=[SERVER])
    try:
        async with stdio_client(params, errlog=errlog) as (reader, writer):
            async with ClientSession(reader, writer) as session:
                await session.initialize()
                yield session
    finally:
        try: errlog.close()
        except Exception: pass


def mcp_result(resp):
    """Extrae el resultado de una llamada a tool de MCP de forma robusta,
    cubriendo distintas versiones del protocolo."""
    for attr in ("structuredContent", "structured_content"):
        sc = getattr(resp, attr, None)
        if sc:
            # Algunas versiones envuelven la salida en {'result': ...}
            if isinstance(sc, dict) and set(sc.keys()) == {"result"}:
                return sc["result"]
            return sc
    # Fallback: parsear texto
    for block in getattr(resp, "content", []) or []:
        txt = getattr(block, "text", None)
        if txt:
            try:    return json.loads(txt)
            except: return {"text": txt}
    return {}


def run_async(coro):
    """Ejecuta una corrutina y desempaca ExceptionGroup para ver el error real."""
    try:
        return asyncio.run(coro)
    except BaseException as e:
        excs = getattr(e, "exceptions", None)
        if excs:
            print("=== ERRORES INTERNOS ===")
            for i, sub in enumerate(excs):
                print(f"\n--- sub-excepcion {i+1} ---")
                traceback.print_exception(type(sub), sub, sub.__traceback__)
        else:
            traceback.print_exception(type(e), e, e.__traceback__)


print("Helper listo: open_mcp_session(), mcp_result(), run_async()")

Helper listo: open_mcp_session(), mcp_result(), run_async()


In [20]:
# ========================================================================
#  RE-PRUEBA - listar capacidades del servidor MCP usando el helper
# ========================================================================
# Requiere que ya se haya ejecutado la Celda 25b (helper open_mcp_session).
# ========================================================================

import asyncio

async def listar_capacidades():
    async with open_mcp_session() as session:
        tools     = await session.list_tools()
        resources = await session.list_resources()
        prompts   = await session.list_prompts()

        print("=" * 60)
        print(f"TOOLS ({len(tools.tools)}):")
        for t in tools.tools:
            desc = (t.description or '').split('\n')[0][:70]
            print(f"  - {t.name:25s} {desc}")

        print(f"\nRESOURCES ({len(resources.resources)}):")
        for r in resources.resources:
            print(f"  - {r.uri}")

        print(f"\nPROMPTS ({len(prompts.prompts)}):")
        for p in prompts.prompts:
            print(f"  - {p.name}")

        # prueba rápida de una tool
        r = await session.call_tool("compute_imc",
                                    {"weight_kg": 82, "height_cm": 180})
        print("\nIMC Andres (82kg, 180cm):", mcp_result(r))


run_async(listar_capacidades())

TOOLS (8):
  - get_client                Devuelve el perfil estructurado de un cliente.
  - compute_imc               Calcula el IMC (kg/m^2) y lo clasifica segun OMS.
  - compute_tmb_tdee          TMB por Mifflin-St Jeor y TDEE segun nivel de actividad.
  - plan_macros               Calcula kcal objetivo y distribucion en gramos de macronutrientes.
  - recommend_exercises       Devuelve ejercicios aptos filtrando los que tengan contraindicaciones.
  - register_measurement      Registra una medida puntual del cliente en el historico.
  - list_progress             Lista el historico de medidas del cliente.
  - detect_risk               Detecta estancamiento o perdida demasiado rapida.

RESOURCES (2):
  - gym://overview
  - exercises://catalog

PROMPTS (2):
  - adaptive_trainer
  - nutrition_coach

IMC Andres (82kg, 180cm): {'imc': 25.31, 'categoria': 'sobrepeso'}


In [21]:
# ========================================================================
#  PARCHE mcp_result: combinar todos los bloques de content cuando MCP

# ========================================================================

import json

def mcp_result(resp):
    """Extrae el resultado de un call_tool() cubriendo todas las variantes:
       - structuredContent con dict/list
       - structuredContent envuelto en {'result': ...}
       - content con UN solo bloque de texto (string o JSON)
       - content con MULTIPLES bloques de texto (cada uno un item de lista)
    """
    # 1) structuredContent (cuando existe es el camino preferido)
    for attr in ("structuredContent", "structured_content"):
        sc = getattr(resp, attr, None)
        if sc is not None:
            if isinstance(sc, dict) and set(sc.keys()) == {"result"}:
                return sc["result"]
            return sc

    # 2) content como bloques
    bloques = getattr(resp, "content", []) or []
    textos = []
    for b in bloques:
        t = getattr(b, "text", None)
        if t:
            textos.append(t)

    if not textos:
        return {}

    # 2a) Un solo bloque -> parsear como JSON si se puede
    if len(textos) == 1:
        t = textos[0]
        try:    return json.loads(t)
        except: return {"text": t}

    # 2b) Multiples bloques -> lista, parseando cada uno como JSON si aplica
    out = []
    for t in textos:
        try:    out.append(json.loads(t))
        except: out.append({"text": t})
    return out


print("mcp_result() parcheado: ahora combina múltiples content blocks.")

mcp_result() parcheado: ahora combina múltiples content blocks.


In [22]:
# ========================================================================
#  Flujo nutricional completo para Valentina (C4)
# ========================================================================

import asyncio

async def probar_flujo_nutricional():
    async with open_mcp_session() as session:
        # 1) Perfil de Valentina (C4)
        resp = await session.call_tool("get_client", {"client_id": "C4"})
        c = mcp_result(resp)
        print("Cliente:", c.get("name"), "-", c.get("goal"))

        # 2) TMB y TDEE con actividad alta
        resp = await session.call_tool("compute_tmb_tdee", {
            "weight_kg": c["weight_kg"],
            "height_cm": c["height_cm"],
            "age":       c["age"],
            "gender":    c["gender"],
            "activity":  "alta",
        })
        tdee = mcp_result(resp)
        print("TMB/TDEE :", tdee)

        # 3) Plan de macros para hipertrofia
        resp = await session.call_tool("plan_macros", {
            "tdee": tdee["tdee"],
            "goal": "hipertrofia",
        })
        macros = mcp_result(resp)
        print("Macros   :", macros)


run_async(probar_flujo_nutricional())

Cliente: Valentina - Aumentar volumen muscular con progreso demostrable
TMB/TDEE : {'tmb': 1343.0, 'tdee': 2316.0, 'factor_actividad': 1.725}
Macros   : {'kcal_objetivo': 2548.0, 'proteina_g': 191.0, 'carbos_g': 287.0, 'grasas_g': 71.0, 'reparto_pct': {'prot': 0.3, 'carb': 0.45, 'gras': 0.25}, 'comidas_por_dia': 4}


In [23]:
# ========================================================================
#  Ejercicios aptos para Mariana (post-lipo, sin impacto)
# ========================================================================

import asyncio

async def probar_ejercicios():
    async with open_mcp_session() as session:
        resp = await session.call_tool("recommend_exercises", {
            "category": "POST_CIRUGIA",
            "restricciones": ["post_cirugia", "rodilla"],
        })
        ejercicios = mcp_result(resp)

        # Por si viene envuelto como dict con 'result'
        if isinstance(ejercicios, dict) and "result" in ejercicios:
            ejercicios = ejercicios["result"]

        print(f"Ejercicios aptos para Mariana (post-lipo): {len(ejercicios)}")
        print("-" * 60)
        for e in ejercicios:
            print(f"  - {e['ejercicio']:30s} "
                  f"(grupo={e['grupo']:10s} "
                  f"nivel={e['nivel']:12s} "
                  f"impacto={e['impacto']})")


run_async(probar_ejercicios())

Ejercicios aptos para Mariana (post-lipo): 12
------------------------------------------------------------
  - prensa_45                      (grupo=piernas    nivel=principiante impacto=bajo)
  - hip_thrust                     (grupo=gluteos    nivel=intermedio   impacto=bajo)
  - remo_polea_baja                (grupo=espalda    nivel=principiante impacto=bajo)
  - press_banca                    (grupo=pecho      nivel=intermedio   impacto=bajo)
  - press_militar                  (grupo=hombro     nivel=intermedio   impacto=bajo)
  - curl_biceps_mancuernas         (grupo=brazos     nivel=principiante impacto=bajo)
  - extension_triceps_polea        (grupo=brazos     nivel=principiante impacto=bajo)
  - eliptica                       (grupo=cardio     nivel=principiante impacto=bajo)
  - bicicleta_estatica             (grupo=cardio     nivel=principiante impacto=bajo)
  - caminadora_inclinada           (grupo=cardio     nivel=principiante impacto=medio)
  - natacion_suave              

# 6. Capa RAG: recuperación aumentada sobre documentos reales

## Flujo del sistema

```
PDFs en docs/
   │
   ▼ (loaders de LangChain)
Documentos crudos (texto + metadatos)
   │
   ▼ (RecursiveCharacterTextSplitter)
Chunks de ~900 caracteres con overlap de 150
   │
   ▼ (HuggingFace: multilingual-e5-small)
Embeddings (vectores de 384 dimensiones, normalizados)
   │
   ▼ (ChromaDB persistente)
Vector store en rag_store/
   │
   ▼ (similarity_search con filtro por metadato)
Top-K chunks relevantes para el LLM
```

## Metadatos por chunk

Cada fragmento conserva:

- `source`       : ruta relativa del archivo original.
- `categoria`    : RENDIMIENTO, NUTRICION, ENTRENADOR o CLIENTES (primer nivel).
- `subcategoria` : POST_CIRUGIA, OBESIDAD, HIPERTROFIA, BAJAR (dentro de CLIENTES).
- `page`         : página del PDF de origen.

Esto permite que el **Entrenador** filtre solo a `categoria=ENTRENADOR`, el **Nutricionista** a `NUTRICION`, y cuando un agente atiende a Mariana (post-lipo) pueda restringirse a `CLIENTES/POST_CIRUGIA`.

## Por qué `multilingual-e5-small`

- Funciona bien en español e inglés (tus PDFs mezclan ambos idiomas).
- Es ligero (384 dimensiones) → rápido en Colab sin GPU.
- Se ejecuta localmente → cero costo por embedding.
- Calidad comparable a `text-embedding-ada-002` en tareas de recuperación.

## Normalización de las consultas (clave para e5)

Los modelos `e5` esperan prefijos explícitos:
- Documentos: `"passage: {texto}"`
- Consultas : `"query: {pregunta}"`

La clase `HuggingFaceEmbeddings` de LangChain permite configurar esto con `embed_instruction` y `query_instruction`.

In [24]:
# ========================================================================
#  Cargar documentos desde docs/ con loaders de LangChain
# ========================================================================
# ========================================================================

from pathlib import Path
from collections import Counter
from langchain_community.document_loaders import (
    PyPDFLoader, TextLoader, Docx2txtLoader, UnstructuredMarkdownLoader
)

DOCS_DIR  = PROJECT_DIR / 'docs'
STORE_DIR = PROJECT_DIR / 'rag_store'
STORE_DIR.mkdir(parents=True, exist_ok=True)


def cargar_uno(path: Path):
    """Devuelve una lista de Documents para un archivo."""
    ext = path.suffix.lower()
    try:
        if ext == '.pdf':   return PyPDFLoader(str(path)).load()
        if ext == '.docx':  return Docx2txtLoader(str(path)).load()
        if ext == '.md':    return UnstructuredMarkdownLoader(str(path)).load()
        if ext == '.txt':   return TextLoader(str(path), encoding='utf-8').load()
    except Exception as e:
        print(f"  [WARN] No se pudo leer {path.name}: {e}")
    return []


todos_los_docs = []
archivos_procesados = 0
archivos_fallidos = 0

for f in sorted(DOCS_DIR.rglob('*')):
    if not f.is_file():
        continue
    docs = cargar_uno(f)
    if not docs:
        archivos_fallidos += 1
        continue

    rel = f.relative_to(DOCS_DIR)
    categoria    = rel.parts[0] if len(rel.parts) > 1 else 'GENERAL'
    subcategoria = rel.parts[1] if len(rel.parts) > 2 else ''

    for d in docs:
        d.metadata['source']       = str(rel)
        d.metadata['categoria']    = categoria
        d.metadata['subcategoria'] = subcategoria

    todos_los_docs.extend(docs)
    archivos_procesados += 1

print(f"Archivos procesados : {archivos_procesados}")
print(f"Archivos fallidos   : {archivos_fallidos}")
print(f"Páginas cargadas    : {len(todos_los_docs)}")

print("\nDistribución por categoría (páginas):")
dist = Counter(d.metadata['categoria'] for d in todos_los_docs)
for cat, n in dist.most_common():
    print(f"  {cat:20s} {n}")

Archivos procesados : 42
Archivos fallidos   : 0
Páginas cargadas    : 4076

Distribución por categoría (páginas):
  CLIENTES             1358
  ENTRENADOR           1315
  NUTRICION            1276
  RENDIMIENTO          127


In [25]:
# ========================================================================
#  Dividir los documentos en chunks de texto
# ========================================================================
# chunk_size  = 900 caracteres (un parrafo tecnico mediano)
# overlap     = 150 caracteres  (evita cortar ideas entre chunks)
# separators  = se intenta cortar por doble salto, luego salto, luego punto, etc.
# ========================================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(todos_los_docs)

print(f"Total de chunks generados: {len(chunks)}")
print("-" * 60)
print("Ejemplo de chunk (primeros 300 caracteres):")
print(chunks[0].page_content[:300], "...")
print("\nMetadata:", chunks[0].metadata)

Total de chunks generados: 13377
------------------------------------------------------------
Ejemplo de chunk (primeros 300 caracteres):
Hoja informativa
Pautas nutricionales  
y ergogenia en el  
incremento de masa  
muscular
Contenidos desarrollados por el Grupo de Especialización 
de Nutrición y Dietética para la Actividad Física y el Deporte 
(GE-NuDAFD)* de la Academia Española de Nutrición y  
Dietética.
T rabajo realizado por  ...

Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2022-09-22T11:30:59+02:00', 'moddate': '2022-09-22T11:31:00+02:00', 'trapped': '/False', 'source': 'CLIENTES/BAJAR/AEDN_2022_nutricion_deportiva_borrador_V0.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'categoria': 'CLIENTES', 'subcategoria': 'BAJAR'}


In [26]:
# == Embeddings + Chroma vector store ===
import os, shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

EMBED_MODEL = "intfloat/multilingual-e5-small"
PERSIST_DIR = "/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/rag_store"

# Limpia cualquier store previo corrupto
if os.path.isdir(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
os.makedirs(PERSIST_DIR, exist_ok=True)

# Embeddings multilingües (compatibles con español)
emb = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Modelo de embeddings cargado: {EMBED_MODEL}")

# Probamos que funcione antes de indexar
vec_prueba = emb.embed_query("rutina de hipertrofia para hombre")
print(f"Dimensión del vector: {len(vec_prueba)}")
print(f"Primeros 5 valores: {vec_prueba[:5]}")

# Crear Chroma store vacío (se poblará en la siguiente celda)
vectorstore = Chroma(
    collection_name="gym_rag",
    embedding_function=emb,
    persist_directory=PERSIST_DIR,
)

print(f"\nVector store inicializado en: {PERSIST_DIR}")
print(f"Documentos actuales en la colección: {vectorstore._collection.count()}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo de embeddings cargado: intfloat/multilingual-e5-small
Dimensión del vector: 384
Primeros 5 valores: [0.03552229329943657, -0.0045393966138362885, -0.08069183677434921, -0.08112915605306625, 0.07032759487628937]

Vector store inicializado en: /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/rag_store
Documentos actuales en la colección: 0


In [27]:
# === Indexar PDFs al vector store con metadatos ===
import os, re, glob, time
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.auto import tqdm

DOCS_DIR = "/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/docs"

# Categorías y subcategorías esperadas según estructura del Drive
CATEGORIAS = {"RENDIMIENTO", "NUTRICION", "ENTRENADOR", "CLIENTES"}
SUBCATEGORIAS = {"POST_CIRUGIA", "OBESIDAD", "HIPERTROFIA", "BAJAR"}

def clasificar_ruta(ruta: str):
    """Extrae categoria y subcategoria de la ruta del archivo."""
    partes = [p.upper() for p in Path(ruta).parts]
    categoria = "GENERAL"
    subcategoria = "NONE"
    for p in partes:
        if p in CATEGORIAS:
            categoria = p
        if p in SUBCATEGORIAS:
            subcategoria = p
    return categoria, subcategoria

# Listar todos los PDFs recursivamente
pdfs = sorted(glob.glob(os.path.join(DOCS_DIR, "**", "*.pdf"), recursive=True))
print(f"Total de PDFs encontrados: {len(pdfs)}")

# Splitter robusto (separadores típicos de documentos en español)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""],
)

total_chunks = 0
fallidos = []
t0 = time.time()

for ruta in tqdm(pdfs, desc="Indexando PDFs"):
    try:
        cat, sub = clasificar_ruta(ruta)
        loader = PyPDFLoader(ruta)
        paginas = loader.load()
        if not paginas:
            continue
        chunks = splitter.split_documents(paginas)
        # Enriquecer metadatos
        for ch in chunks:
            ch.metadata["categoria"] = cat
            ch.metadata["subcategoria"] = sub
            ch.metadata["archivo"] = os.path.basename(ruta)
            # Chroma no acepta valores None en metadatos
            for k, v in list(ch.metadata.items()):
                if v is None:
                    ch.metadata[k] = ""
        if chunks:
            vectorstore.add_documents(chunks)
            total_chunks += len(chunks)
    except Exception as e:
        fallidos.append((os.path.basename(ruta), str(e)[:120]))

print(f"\nIndexado completo en {time.time()-t0:.1f}s")
print(f"Chunks totales: {total_chunks}")
print(f"Documentos en la colección: {vectorstore._collection.count()}")
if fallidos:
    print(f"\nPDFs con problemas ({len(fallidos)}):")
    for f, err in fallidos[:10]:
        print(f"  - {f}: {err}")

Total de PDFs encontrados: 42


Indexando PDFs:   0%|          | 0/42 [00:00<?, ?it/s]


Indexado completo en 320.0s
Chunks totales: 14832
Documentos en la colección: 14832


# 7. Capa de agentes: LLMs con acceso a tools (MCP + RAG)

## Flujo del sistema

```
Mensaje del usuario
   │
   ▼ (system prompt + user message)
GPT-4o (agente especialista)
   │
   ▼ (tool_choice=auto)
Decisión: ¿necesito tools?
   │
   ├── NO ──▶ respuesta final en lenguaje natural
   │
   └── SI ──▶ tool_call(name, args)
              │
              ▼ (dispatch_tool)
              ┌─────────────┬──────────────┐
              ▼             ▼              ▼
          MCP stdio     rag_search    (futuras tools)
          (cálculos,   (recuperación   ...
           perfiles)    documental)
              │             │
              └──────┬──────┘
                     ▼
              resultado JSON → role="tool"
                     │
                     ▼
            (vuelve al loop hasta respuesta final)
```

## Piezas del sistema

- `TOOLS_SCHEMA` — catálogo **autogenerado** desde el servidor MCP con
  `list_tools()`, más la entrada local `rag_search`. Siempre sincronizado con
  las firmas reales del servidor, sin duplicación manual.
- `dispatch_tool(name, args)` — router único que envía la llamada al MCP
  (vía `open_mcp_session`) o a la función RAG local, y serializa la respuesta
  como JSON para devolverla al LLM.
- `agent_run(system_prompt, user_message, tools, ...)` — loop canónico de
  tool-calling: `LLM → tool_calls → dispatch → tool results → LLM → ...` hasta
  que el modelo emita respuesta final o se alcance `max_iters`.

## Los cuatro roles

Cada agente es el **mismo** loop `agent_run` con distinto system prompt y,
opcionalmente, distinto modelo. Se definen como constantes:

- `SYSTEM_ENTRENADOR`     — diseña rutinas, respeta restricciones, fundamenta
                            con `rag_search(categoria=ENTRENADOR o CLIENTES)`.
- `SYSTEM_NUTRICIONISTA`  — calcula TDEE + macros, cita RAG de `NUTRICION`.
- `SYSTEM_ANALISTA`       — línea base (IMC, TMB, TDEE), registro de mediciones,
                            detección de riesgos, progreso.
- `SYSTEM_COORDINADOR`    — orquesta a los anteriores como sub-agentes
                            (patrón **agent-as-tool**) y sintetiza.

## Por qué este diseño

- **Separación de preocupaciones**: el LLM razona y decide; el MCP calcula con
  precisión; el RAG recupera evidencia. Ningún componente intenta hacer el
  trabajo de otro.
- **Interoperabilidad**: cualquier otro cliente compatible con MCP (Claude,
  Cursor, otro agente) puede consumir las mismas tools sin modificar el servidor.
- **Trazabilidad**: cada tool-call queda registrada y es auditable — requisito
  para aplicaciones de salud y deporte donde se debe justificar cada decisión.
- **Coste controlado**: solo el Coordinador usa `gpt-4o`; los especialistas
  pueden migrarse a modelos más pequeños o fine-tuned sin tocar la orquestación.

In [28]:
# === Función rag_search ===
from typing import Optional, List, Dict

def rag_search(
    query: str,
    k: int = 4,
    categoria: Optional[str] = None,
    subcategoria: Optional[str] = None,
) -> List[Dict]:
    """
    Busca en el vector store con filtros opcionales.

    Args:
        query: pregunta o tema a buscar
        k: número de resultados
        categoria: filtra por RENDIMIENTO, NUTRICION, ENTRENADOR o CLIENTES
        subcategoria: filtra por POST_CIRUGIA, OBESIDAD, HIPERTROFIA o BAJAR

    Returns:
        Lista de dicts con {texto, archivo, categoria, subcategoria, score}
    """
    # Construir filtro de metadatos (formato Chroma)
    filtros = []
    if categoria:
        filtros.append({"categoria": categoria.upper()})
    if subcategoria:
        filtros.append({"subcategoria": subcategoria.upper()})

    if len(filtros) == 0:
        where = None
    elif len(filtros) == 1:
        where = filtros[0]
    else:
        where = {"$and": filtros}

    resultados = vectorstore.similarity_search_with_score(
        query=query,
        k=k,
        filter=where,
    )

    salida = []
    for doc, score in resultados:
        salida.append({
            "texto": doc.page_content.strip(),
            "archivo": doc.metadata.get("archivo", ""),
            "categoria": doc.metadata.get("categoria", ""),
            "subcategoria": doc.metadata.get("subcategoria", ""),
            "pagina": doc.metadata.get("page", -1),
            "score": round(float(score), 4),
        })
    return salida


def mostrar_resultados(resultados, max_chars=280):
    """Imprime resultados de rag_search de forma legible."""
    if not resultados:
        print("(sin resultados)")
        return
    for i, r in enumerate(resultados, 1):
        print(f"[{i}] {r['archivo']} | cat={r['categoria']} | sub={r['subcategoria']} | p.{r['pagina']} | score={r['score']}")
        txt = r["texto"].replace("\n", " ")
        print(f"    {txt[:max_chars]}{'...' if len(txt) > max_chars else ''}\n")

print("Función rag_search lista.")

Función rag_search lista.


In [29]:
# === Tests de RAG ===

print("=" * 70)
print("TEST 1: Consulta general sobre hipertrofia (sin filtro)")
print("=" * 70)
mostrar_resultados(rag_search("rutina de hipertrofia para ganar masa muscular", k=3))

print("=" * 70)
print("TEST 2: Nutrición específica (filtro por categoría NUTRICION)")
print("=" * 70)
mostrar_resultados(rag_search(
    "distribución de macronutrientes y proteína para definición",
    k=3,
    categoria="NUTRICION",
))

print("=" * 70)
print("TEST 3: Cliente con sobrepeso y problemas de rodilla (CLIENTES + BAJAR)")
print("=" * 70)
mostrar_resultados(rag_search(
    "ejercicios sin impacto para persona con sobrepeso y problemas de rodilla",
    k=3,
    categoria="CLIENTES",
    subcategoria="BAJAR",
))

print("=" * 70)
print("TEST 4: Postoperatorio lipoescultura (CLIENTES + POST_CIRUGIA)")
print("=" * 70)
mostrar_resultados(rag_search(
    "recomendaciones de ejercicio después de lipoescultura para mantener resultados",
    k=3,
    categoria="CLIENTES",
    subcategoria="POST_CIRUGIA",
))

TEST 1: Consulta general sobre hipertrofia (sin filtro)
[1] rev02_raya.pdf | cat=CLIENTES | sub=HIPERTROFIA | p.0 | score=0.2119
    Palabras clave:   Hipertrofia.   Entrenamiento con cargas.   Nutrición. Summary Introduction: The increase of the muscle mass is one of the main challenges of the athletic trainers, either to optimize the  performance, for esthetical reasons or for the health improvement. Therefo...

[2] rev02_raya.pdf | cat=ENTRENADOR | sub=NONE | p.0 | score=0.2119
    Palabras clave:   Hipertrofia.   Entrenamiento con cargas.   Nutrición. Summary Introduction: The increase of the muscle mass is one of the main challenges of the athletic trainers, either to optimize the  performance, for esthetical reasons or for the health improvement. Therefo...

[3] rev02_raya.pdf | cat=CLIENTES | sub=HIPERTROFIA | p.3 | score=0.2213
    muscular de 3 o 6 días. Schoenfeld  et al. (2015) Frecuencia n=20 jóvenes varo- nes entrena- dos 19-29 años Asignación aleatoria a uno de los dos pr

In [30]:
#  Autogenerar TOOLS_SCHEMA desde el MCP ===

async def _fetch_mcp_tools():
    async with open_mcp_session() as session:
        resp = await session.list_tools()
        return resp.tools

_mcp_tools = run_async(_fetch_mcp_tools())

TOOLS_SCHEMA = []
MCP_TOOL_NAMES = set()

for t in _mcp_tools:
    schema = getattr(t, "inputSchema", None) or {"type": "object", "properties": {}}
    TOOLS_SCHEMA.append({
        "type": "function",
        "function": {
            "name": t.name,
            "description": (t.description or "").strip() or f"Tool MCP {t.name}",
            "parameters": schema,
        },
    })
    MCP_TOOL_NAMES.add(t.name)

# Añadir rag_search (tool local, no MCP)
TOOLS_SCHEMA.append({
    "type": "function",
    "function": {
        "name": "rag_search",
        "description": (
            "Busca evidencia en la base documental del gimnasio (PDFs de entrenamiento, "
            "nutrición, rendimiento y perfiles de cliente). Úsala SIEMPRE que necesites "
            "fundamentar una recomendación con material de referencia."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "k": {"type": "integer", "default": 4},
                "categoria": {
                    "type": "string",
                    "enum": ["RENDIMIENTO", "NUTRICION", "ENTRENADOR", "CLIENTES"],
                },
                "subcategoria": {
                    "type": "string",
                    "enum": ["POST_CIRUGIA", "OBESIDAD", "HIPERTROFIA", "BAJAR"],
                },
            },
            "required": ["query"],
        },
    },
})

print(f"Tools MCP detectadas: {len(MCP_TOOL_NAMES)}")
for t in TOOLS_SCHEMA:
    name = t["function"]["name"]
    props = list(t["function"]["parameters"].get("properties", {}).keys())
    print(f"  - {name}({', '.join(props)})")

Tools MCP detectadas: 8
  - get_client(client_id)
  - compute_imc(weight_kg, height_cm)
  - compute_tmb_tdee(weight_kg, height_cm, age, gender, activity)
  - plan_macros(tdee, goal)
  - recommend_exercises(category, restricciones)
  - register_measurement(client_id, weight_kg, body_fat_pct, note)
  - list_progress(client_id)
  - detect_risk(client_id)
  - rag_search(query, k, categoria, subcategoria)


In [31]:
# Dispatcher de tools ===
import json
from typing import Any, Dict

async def _call_mcp_tool(name: str, args: Dict[str, Any]) -> Any:
    async with open_mcp_session() as session:
        resp = await session.call_tool(name, args)
        return mcp_result(resp)

def dispatch_tool(name: str, args: Dict[str, Any]) -> str:
    """Ejecuta la tool y devuelve el resultado serializado como string."""
    try:
        if name in MCP_TOOL_NAMES:
            resultado = run_async(_call_mcp_tool(name, args))
        elif name == "rag_search":
            resultado = rag_search(**args)
            resultado = [
                {
                    "texto": r["texto"][:600],
                    "archivo": r["archivo"],
                    "categoria": r["categoria"],
                    "subcategoria": r["subcategoria"],
                    "score": r["score"],
                }
                for r in resultado
            ]
        else:
            return json.dumps({"error": f"Tool desconocida: {name}"}, ensure_ascii=False)
        return json.dumps(resultado, ensure_ascii=False, default=str)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"}, ensure_ascii=False)

# Prueba con los nombres REALES de los parámetros (inspecciona la celda 35)
print("Prueba get_client(C2):")
print(dispatch_tool("get_client", {"client_id": "C2"})[:300], "\n")

# Usa los nombres que aparezcan en la celda 35; probable sea weight_kg / height_cm
print("Prueba compute_imc(weight_kg=82, height_cm=180):")
print(dispatch_tool("compute_imc", {"weight_kg": 82, "height_cm": 180}))

Prueba get_client(C2):
{"id": "C2", "name": "Andres", "gender": "M", "age": 29, "height_cm": 180, "weight_kg": 82, "body_fat_pct": 14.9, "category": "HIPERTROFIA", "goal": "Aumentar masa muscular y lograr mayor definicion", "consistency": "alta", "discipline": "baja", "weekly_frequency": 5, "restrictions": [], "notes": "T 

Prueba compute_imc(weight_kg=82, height_cm=180):
{"imc": 25.31, "categoria": "sobrepeso"}


In [32]:
# === CELDA AUX: Recargar OPENAI_API_KEY ===
import os, getpass
from openai import OpenAI

# Opción 1: pedirla por input seguro (recomendado)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Pega tu OPENAI_API_KEY: ").strip()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
assert OPENAI_API_KEY.startswith("sk-"), "La clave no parece válida (debería empezar con 'sk-')"

client_oai = OpenAI(api_key=OPENAI_API_KEY)

# Ping rápido para confirmar
resp = client_oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Responde exactamente: OK"}],
    max_tokens=5,
)
print("Respuesta:", resp.choices[0].message.content)
print("OPENAI_API_KEY cargada correctamente.")

Respuesta: OK
OPENAI_API_KEY cargada correctamente.


## PASO 8 — Arquitectura multi-agente con tool-calling

Transformamos las llamadas simples a GPT-4 en **agentes con herramientas**.
Cada agente recibe un catálogo (tools del MCP + `rag_search`) y GPT-4o decide
autónomamente qué tool llamar, con qué argumentos, en qué orden y cómo
combinar resultados.

**Piezas clave:**
- `TOOLS_SCHEMA`: autogenerado desde el MCP con `list_tools()`, garantiza que el
  schema de OpenAI siempre esté sincronizado con el servidor.
- `dispatch_tool(name, args)`: ruta unificada que enruta la llamada a MCP o a
  la función RAG local y devuelve la respuesta serializada en JSON.
- `agent_run(system_prompt, user_message, tools, ...)`: loop canónico de
  tool-calling (llamar → ejecutar tools → alimentar respuestas → repetir hasta
  que el LLM emita respuesta final).

Se definen cuatro roles con system-prompts diferenciados:
**Entrenador**, **Nutricionista**, **Analista de rendimiento** y **Coordinador**.
Cada uno tiene reglas de dominio y un estilo de salida esperado.

In [33]:
# ===Loop genérico de agente con tool-calling ===
import json
from openai import OpenAI

client_oai = OpenAI(api_key=OPENAI_API_KEY)

def agent_run(
    system_prompt: str,
    user_message: str,
    tools=TOOLS_SCHEMA,
    model: str = "gpt-4o",
    max_iters: int = 6,
    verbose: bool = True,
    history: list = None,
) -> dict:
    """
    Ejecuta un agente con tool-calling hasta que emita una respuesta final.
    Devuelve {"answer": str, "messages": [...], "trace": [...]}.
    """
    messages = history[:] if history else []
    if not messages or messages[0].get("role") != "system":
        messages = [{"role": "system", "content": system_prompt}] + messages
    messages.append({"role": "user", "content": user_message})

    trace = []

    for it in range(max_iters):
        resp = client_oai.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0.3,
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"\n[iter {it+1}] respuesta final.")
            return {"answer": msg.content or "", "messages": messages, "trace": trace}

        # Ejecutar todas las tool_calls que pidió el modelo
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except Exception:
                args = {}
            if verbose:
                print(f"[iter {it+1}] → tool: {name}({args})")
            result = dispatch_tool(name, args)
            trace.append({"tool": name, "args": args, "result": result[:500]})
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    if verbose:
        print(f"\n[warn] alcanzado max_iters={max_iters} sin respuesta final.")
    return {"answer": "(sin respuesta final)", "messages": messages, "trace": trace}

print("agent_run lista.")

agent_run lista.


In [34]:
# === System prompts de los agentes ===

SYSTEM_ENTRENADOR = """Eres el AGENTE ENTRENADOR de FitCoach, licenciado en educación física con especialización en acondicionamiento.

Tu trabajo:
- Diseñar rutinas y prescripciones de ejercicio adaptadas al cliente.
- SIEMPRE usar get_client para obtener el perfil antes de recomendar.
- SIEMPRE usar recommend_exercises para la selección base.
- Respetar restricciones médicas (p.ej. evitar impacto en rodilla) y post-operatorios.
- Usar rag_search (categoria ENTRENADOR o CLIENTES) para fundamentar con la bibliografía del gimnasio.
- Responder en español, con formato claro: objetivo, rutina semanal, progresión, advertencias.

No inventes datos del cliente: si te falta información, pídela o consulta get_client.
"""

SYSTEM_NUTRICIONISTA = """Eres el AGENTE NUTRICIONISTA de FitCoach, profesional en nutrición deportiva.

Tu trabajo:
- Diseñar planes nutricionales alineados al objetivo del cliente.
- SIEMPRE calcular TDEE con compute_tmb_tdee y macros con plan_macros.
- Consultar get_client antes de emitir recomendaciones.
- Usar rag_search (categoria NUTRICION) para sustentar distribución de macros y timing.
- Considerar condiciones como post-operatorio, sobrepeso, o bajo peso.
- Entregar: calorías objetivo, macros en gramos, ejemplos de comidas, hidratación.

Responde en español. Sé concreto con cifras y evita generalidades.
"""

SYSTEM_ANALISTA = """Eres el AGENTE ANALISTA DE RENDIMIENTO de FitCoach.

Tu trabajo:
- Establecer línea base del cliente (peso, talla, % grasa, IMC).
- Registrar mediciones (register_measurement) y analizar progreso (list_progress).
- Detectar riesgos con detect_risk y alertar al coordinador.
- Usar rag_search (categoria RENDIMIENTO) para respaldar protocolos de evaluación.
- Presentar resultados con números, tendencias y recomendaciones objetivas.

Responde en español y siempre con tablas/listas numéricas cuando aplique.
"""

SYSTEM_COORDINADOR = """Eres el AGENTE COORDINADOR de FitCoach, responsable de orquestar la atención multi-agente al cliente.

Tu flujo de trabajo:
1. Identifica el CLIENT_ID mencionado (C1, C2, C3, C4) y usa get_client para conocerlo.
2. Decide qué especialistas intervenir: Entrenador, Nutricionista, Analista, o varios.
3. Usa las tools directamente para recolectar datos (IMC, TDEE, riesgos, progreso, RAG).
4. Sintetiza una respuesta integral al cliente: diagnóstico breve, plan de entrenamiento,
   plan nutricional, seguimiento sugerido, y advertencias/riesgos.
5. Cita la evidencia RAG cuando la uses (archivo + score).

Sé profesional, claro, estructurado. Responde siempre en español.
"""

print("System prompts definidos para los 4 agentes.")

System prompts definidos para los 4 agentes.


In [35]:
# === Smoke test del Analista sobre cliente C3 ===

out = agent_run(
    system_prompt=SYSTEM_ANALISTA,
    user_message="Haz la evaluación inicial del cliente C3 (Jorge): IMC, clasificación y riesgos.",
    verbose=True,
)

print("\n" + "=" * 70)
print("RESPUESTA FINAL DEL ANALISTA")
print("=" * 70)
print(out["answer"])

[iter 1] → tool: get_client({'client_id': 'C3'})
[iter 2] → tool: compute_imc({'weight_kg': 95, 'height_cm': 172})
[iter 2] → tool: detect_risk({'client_id': 'C3'})

[iter 3] respuesta final.

RESPUESTA FINAL DEL ANALISTA
### Evaluación Inicial de Jorge (Cliente C3)

1. **Datos Básicos:**
   - Edad: 55 años
   - Altura: 172 cm
   - Peso: 95 kg
   - % Grasa Corporal: 32.0%

2. **IMC y Clasificación:**
   - **IMC:** 32.11
   - **Clasificación según OMS:** Obesidad

3. **Riesgos Detectados:**
   - Actualmente, hay **insuficientes datos** para detectar riesgos específicos de estancamiento o pérdida de peso demasiado rápida. Es importante realizar un seguimiento regular de las medidas para evaluar el progreso y ajustar el plan según sea necesario.

### Recomendaciones
- **Confirmación de Medidas:** Es crucial confirmar el peso, altura y porcentaje de grasa en la próxima cita, como se indica en las notas del cliente.
- **Plan de Acción:** Dado que Jorge tiene restricciones como "sin impacto 

## PASO 9 — Memoria episódica por cliente (lifelong learning)

Para cumplir con el requisito de **aprendizaje continuo** del proyecto, cada
cliente tiene un archivo `memory/C{id}.jsonl` append-only donde se registran:
- Mensajes del usuario
- Tool-calls realizadas (nombre, args, resumen del resultado)
- Respuestas finales del asistente

Al iniciar una nueva interacción, el agente carga un **resumen** de los últimos
20 episodios y lo inyecta como contexto en el system prompt. Así, en la segunda
visita de Mariana el Analista no vuelve a pedir su línea base: la recuerda.

La función `agent_run_with_memory(client_id, ...)` envuelve a `agent_run`
añadiendo la carga del historial y la persistencia del episodio. Este diseño
es simple, auditable y suficiente para la escala del proyecto (un gimnasio).

In [36]:
# ===  Memoria episódica por cliente ===
import json, os
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict

MEMORY_DIR = "/content/proyecto_entrenador/memory"
os.makedirs(MEMORY_DIR, exist_ok=True)

def _memory_path(client_id: str) -> str:
    return os.path.join(MEMORY_DIR, f"{client_id.upper()}.jsonl")

def save_episode(
    client_id: str,
    role: str,
    content: str,
    agent: Optional[str] = None,
    metadata: Optional[dict] = None,
):
    """Append-only: guarda un episodio en la memoria del cliente."""
    episodio = {
        "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "client_id": client_id.upper(),
        "role": role,
        "agent": agent or "",
        "content": content,
        "metadata": metadata or {},
    }
    with open(_memory_path(client_id), "a", encoding="utf-8") as f:
        f.write(json.dumps(episodio, ensure_ascii=False) + "\n")
    return episodio

def load_recent_episodes(client_id: str, n: int = 10) -> List[Dict]:
    """Carga los últimos n episodios del cliente (más reciente al final)."""
    path = _memory_path(client_id)
    if not os.path.isfile(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        lineas = f.readlines()
    episodios = []
    for ln in lineas[-n:]:
        try:
            episodios.append(json.loads(ln))
        except json.JSONDecodeError:
            continue
    return episodios

def memory_summary(client_id: str) -> str:
    """Resumen textual compacto de la memoria (para inyectar como contexto)."""
    eps = load_recent_episodes(client_id, n=20)
    if not eps:
        return "(sin historial previo)"
    lineas = []
    for e in eps:
        ts = e["ts"][:10]
        rol = e["role"]
        ag = e.get("agent") or ""
        txt = e["content"].replace("\n", " ").strip()
        if len(txt) > 200:
            txt = txt[:200] + "..."
        etiqueta = f"{ts} [{rol}{':'+ag if ag else ''}]"
        lineas.append(f"{etiqueta} {txt}")
    return "\n".join(lineas)

def clear_memory(client_id: str):
    """Borra la memoria del cliente (útil para tests)."""
    path = _memory_path(client_id)
    if os.path.isfile(path):
        os.remove(path)
    print(f"Memoria de {client_id} borrada.")

print("Módulo de memoria episódica cargado.")
print(f"Directorio: {MEMORY_DIR}")

Módulo de memoria episódica cargado.
Directorio: /content/proyecto_entrenador/memory


In [37]:
# === Agente con memoria episódica ===

def agent_run_with_memory(
    client_id: str,
    system_prompt: str,
    user_message: str,
    agent_name: str = "coordinador",
    tools=TOOLS_SCHEMA,
    model: str = "gpt-4o",
    max_iters: int = 6,
    verbose: bool = True,
) -> dict:
    """Ejecuta un agente con contexto de memoria y guarda el episodio."""

    # 1. Construir system prompt extendido con memoria previa
    resumen = memory_summary(client_id)
    sys_extended = (
        system_prompt
        + f"\n\n--- HISTORIAL DEL CLIENTE {client_id} ---\n"
        + resumen
        + "\n--- FIN HISTORIAL ---\n"
        "Usa este historial para dar continuidad. No repitas análisis previos; refiérete a ellos."
    )

    # 2. Guardar el mensaje del usuario
    save_episode(client_id, role="user", content=user_message, agent=agent_name)

    # 3. Ejecutar el agente
    out = agent_run(
        system_prompt=sys_extended,
        user_message=user_message,
        tools=tools,
        model=model,
        max_iters=max_iters,
        verbose=verbose,
    )

    # 4. Guardar traza de tools y respuesta final
    for t in out.get("trace", []):
        save_episode(
            client_id,
            role="tool",
            content=json.dumps({"tool": t["tool"], "args": t["args"]}, ensure_ascii=False),
            agent=agent_name,
            metadata={"result_preview": t["result"][:200]},
        )
    save_episode(client_id, role="assistant", content=out["answer"], agent=agent_name)

    return out

print("agent_run_with_memory lista.")

agent_run_with_memory lista.


In [38]:
# === Test de memoria episódica con C1 ===

# Limpiamos historial previo para un test limpio
clear_memory("C1")

# --- Turno 1: evaluación inicial ---
print("=" * 70)
print("TURNO 1 — Evaluación inicial de Mariana (C1)")
print("=" * 70)
out1 = agent_run_with_memory(
    client_id="C1",
    system_prompt=SYSTEM_ANALISTA,
    user_message="Haz la evaluación inicial de C1 y calcula su IMC.",
    agent_name="analista",
    verbose=True,
)
print("\n" + out1["answer"])

# --- Turno 2: se referencia lo anterior ---
print("\n" + "=" * 70)
print("TURNO 2 — Con memoria del turno 1")
print("=" * 70)
out2 = agent_run_with_memory(
    client_id="C1",
    system_prompt=SYSTEM_ANALISTA,
    user_message="Dada la evaluación previa, ¿qué medición de seguimiento recomiendas en 4 semanas?",
    agent_name="analista",
    verbose=True,
)
print("\n" + out2["answer"])

# Inspeccionar memoria persistida
print("\n" + "=" * 70)
print("CONTENIDO DE MEMORIA C1 (últimos episodios)")
print("=" * 70)
for ep in load_recent_episodes("C1", n=20):
    txt = ep["content"]
    if len(txt) > 180:
        txt = txt[:180] + "..."
    print(f"[{ep['ts']}] {ep['role']:9} ({ep['agent']:12}) → {txt}")

Memoria de C1 borrada.
TURNO 1 — Evaluación inicial de Mariana (C1)


/tmp/ipykernel_6527/2922189501.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",


[iter 1] → tool: get_client({'client_id': 'C1'})
[iter 2] → tool: compute_imc({'weight_kg': 58, 'height_cm': 165})

[iter 3] respuesta final.

### Evaluación Inicial de Mariana (C1)

1. **Datos Básicos:**
   - **Edad:** 25 años
   - **Género:** Femenino
   - **Altura:** 165 cm
   - **Peso:** 58 kg
   - **% Grasa Corporal:** 22.0%
   - **Categoría:** Post-cirugía
   - **Objetivo:** Mantener medidas obtenidas tras lipo-escultura
   - **Frecuencia Semanal de Ejercicio:** 3 veces
   - **Consistencia y Disciplina:** Media

2. **Restricciones:**
   - Sin impacto las primeras 6 semanas
   - Evitar abdominales con carga hasta la 4ª semana
   - Uso de prenda de compresión

3. **Notas Adicionales:**
   - Post-operatorio de lipo-escultura en recuperación. Datos iniciales estimados.

4. **Índice de Masa Corporal (IMC):**
   - **IMC Calculado:** 21.3
   - **Categoría según OMS:** Normal

Mariana se encuentra en un rango de IMC normal, lo cual es adecuado para su objetivo de mantener las medidas pos

/tmp/ipykernel_6527/2922189501.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",



[iter 1] respuesta final.

Para realizar un seguimiento efectivo del progreso de Mariana (C1) en 4 semanas, recomiendo la siguiente medición:

1. **Peso Corporal (kg):** Para evaluar cambios en el peso.
2. **% Grasa Corporal:** Para analizar cambios en la composición corporal.
3. **IMC:** Aunque ya calculado, es útil para observar tendencias.

Estas mediciones permitirán evaluar si hay progreso hacia sus objetivos y detectar cualquier posible estancamiento o cambio inesperado. Además, se puede complementar con una evaluación de su nivel de actividad y ajuste de su plan de entrenamiento o nutrición si es necesario.

CONTENIDO DE MEMORIA C1 (últimos episodios)
[2026-04-26T20:29:03Z] user      (analista    ) → Haz la evaluación inicial de C1 y calcula su IMC.
[2026-04-26T20:29:50Z] tool      (analista    ) → {"tool": "get_client", "args": {"client_id": "C1"}}
[2026-04-26T20:29:50Z] tool      (analista    ) → {"tool": "compute_imc", "args": {"weight_kg": 58, "height_cm": 165}}
[2026-04-26

In [39]:
# === Sub-agentes como tools del Coordinador ===

def _ask_specialist(system_prompt: str, task: str, client_id: str = None, model="gpt-4o"):
    """Ejecuta un especialista y devuelve su respuesta como string."""
    # Si hay client_id, lo prepend para que el especialista lo use
    user_msg = task if not client_id else f"[Cliente: {client_id}]\n{task}"
    out = agent_run(
        system_prompt=system_prompt,
        user_message=user_msg,
        tools=TOOLS_SCHEMA,
        model=model,
        max_iters=6,
        verbose=False,
    )
    return out["answer"]

def ask_trainer(client_id: str, task: str) -> str:
    return _ask_specialist(SYSTEM_ENTRENADOR, task, client_id)

def ask_nutritionist(client_id: str, task: str) -> str:
    return _ask_specialist(SYSTEM_NUTRICIONISTA, task, client_id)

def ask_analyst(client_id: str, task: str) -> str:
    return _ask_specialist(SYSTEM_ANALISTA, task, client_id)

print("Sub-agentes listos: ask_trainer, ask_nutritionist, ask_analyst")

Sub-agentes listos: ask_trainer, ask_nutritionist, ask_analyst


In [40]:
# === Catálogo y dispatcher extendidos para el Coordinador ===

SUBAGENT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "ask_trainer",
            "description": (
                "Delega al AGENTE ENTRENADOR (licenciado en educación física). "
                "Úsalo para diseñar rutinas, prescripción de ejercicio, progresión y adaptaciones."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "client_id": {"type": "string"},
                    "task": {"type": "string", "description": "Pregunta o misión específica para el Entrenador."},
                },
                "required": ["client_id", "task"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ask_nutritionist",
            "description": (
                "Delega al AGENTE NUTRICIONISTA deportivo. "
                "Úsalo para plan calórico, macros, timing de comidas, hidratación."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "client_id": {"type": "string"},
                    "task": {"type": "string"},
                },
                "required": ["client_id", "task"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ask_analyst",
            "description": (
                "Delega al AGENTE ANALISTA DE RENDIMIENTO. "
                "Úsalo para línea base, IMC/TMB/TDEE, registro de mediciones, progreso y riesgos."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "client_id": {"type": "string"},
                    "task": {"type": "string"},
                },
                "required": ["client_id", "task"],
            },
        },
    },
]

COORDINATOR_TOOLS_SCHEMA = TOOLS_SCHEMA + SUBAGENT_TOOLS
SUBAGENT_NAMES = {"ask_trainer", "ask_nutritionist", "ask_analyst"}

def dispatch_tool_coord(name: str, args: Dict[str, Any]) -> str:
    """Dispatcher extendido: soporta sub-agentes además de MCP + RAG."""
    try:
        if name in SUBAGENT_NAMES:
            fn = {
                "ask_trainer": ask_trainer,
                "ask_nutritionist": ask_nutritionist,
                "ask_analyst": ask_analyst,
            }[name]
            resultado = fn(**args)
            return json.dumps({"agent_response": resultado}, ensure_ascii=False)
        return dispatch_tool(name, args)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"}, ensure_ascii=False)

# Variante de agent_run que usa el dispatcher extendido
def coordinator_run(
    user_message: str,
    client_id: str,
    model: str = "gpt-4o",
    max_iters: int = 8,
    verbose: bool = True,
) -> dict:
    """Loop del Coordinador con acceso a sub-agentes, MCP y RAG."""
    sys_ext = (
        SYSTEM_COORDINADOR
        + f"\n\n--- HISTORIAL DEL CLIENTE {client_id} ---\n"
        + memory_summary(client_id)
        + "\n--- FIN HISTORIAL ---\n"
    )
    messages = [
        {"role": "system", "content": sys_ext},
        {"role": "user", "content": user_message},
    ]
    trace = []
    save_episode(client_id, "user", user_message, agent="coordinador")

    for it in range(max_iters):
        resp = client_oai.chat.completions.create(
            model=model,
            messages=messages,
            tools=COORDINATOR_TOOLS_SCHEMA,
            tool_choice="auto",
            temperature=0.3,
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"\n[coord iter {it+1}] respuesta final.")
            save_episode(client_id, "assistant", msg.content or "", agent="coordinador")
            return {"answer": msg.content or "", "messages": messages, "trace": trace}

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except Exception:
                args = {}
            if verbose:
                etq = "SUBAGENTE" if name in SUBAGENT_NAMES else "tool"
                print(f"[coord iter {it+1}] → {etq}: {name}({ {k: (v[:60]+'...' if isinstance(v,str) and len(v)>60 else v) for k,v in args.items()} })")
            result = dispatch_tool_coord(name, args)
            trace.append({"tool": name, "args": args, "result": result[:500]})
            save_episode(
                client_id, "tool",
                content=json.dumps({"tool": name, "args": args}, ensure_ascii=False),
                agent="coordinador",
                metadata={"result_preview": result[:300]},
            )
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    save_episode(client_id, "assistant", "(max_iters alcanzado)", agent="coordinador")
    return {"answer": "(sin respuesta final)", "messages": messages, "trace": trace}

print("coordinator_run lista.")
print(f"Tools disponibles al Coordinador: {len(COORDINATOR_TOOLS_SCHEMA)}")

coordinator_run lista.
Tools disponibles al Coordinador: 12


## PASO 10 — Demostración end-to-end: Coordinador orquestando especialistas

El **Coordinador** es el punto de entrada del sistema. Recibe la petición del
usuario y decide dinámicamente a qué especialistas consultar, usando el patrón
**agent-as-tool**: los sub-agentes (`ask_trainer`, `ask_nutritionist`,
`ask_analyst`) se exponen como tools OpenAI estándar.

**Flujo típico:**
1. Usuario envía consulta integral ("Jorge acaba de llegar, dame un plan completo").
2. Coordinador llama `ask_analyst` para línea base y riesgos.
3. Coordinador llama `ask_trainer` pasándole las restricciones detectadas.
4. Coordinador llama `ask_nutritionist` con el TDEE y objetivo.
5. Coordinador sintetiza un informe unificado y lo devuelve al usuario.

Cada especialista corre internamente su propio loop de tool-calling sobre el MCP
y el RAG, totalmente transparente para el Coordinador. El informe final se
persiste como markdown en `reports/` para trazabilidad.

In [42]:
# === Demo end-to-end con Coordinador para C3 (Jorge) ===

clear_memory("C3")  # test limpio

consulta = """Jorge (C3) acaba de llegar al gimnasio. Es paciente referido por su médico
por sobrepeso. Necesito un plan integral de bienvenida que incluya:
1. Evaluación de línea base (IMC, riesgos).
2. Plan de entrenamiento considerando que no puede hacer ejercicios de impacto.
3. Plan nutricional para bajar peso de forma segura.
4. Cronograma de seguimiento para las primeras 8 semanas.
Entrega un informe unificado, profesional y claro, como si fuera para presentarle al paciente y a su médico."""

out = coordinator_run(user_message=consulta, client_id="C3", verbose=True)

print("\n" + "=" * 70)
print("INFORME INTEGRAL — JORGE (C3)")
print("=" * 70)
print(out["answer"])

# Guardar el informe como markdown
informe_path = "/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/reports/C3_informe_inicial.md"
os.makedirs(os.path.dirname(informe_path), exist_ok=True)
with open(informe_path, "w", encoding="utf-8") as f:
    f.write(f"# Informe Integral — Cliente C3 (Jorge)\n\n")
    f.write(f"**Fecha:** {datetime.utcnow().isoformat(timespec='seconds')}Z\n\n")
    f.write(out["answer"])
print(f"\nInforme guardado en: {informe_path}")

Memoria de C3 borrada.


/tmp/ipykernel_6527/2922189501.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",


[coord iter 1] → tool: get_client({'client_id': 'C3'})


/tmp/ipykernel_6527/2922189501.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",


[coord iter 2] → tool: compute_imc({'weight_kg': 95, 'height_cm': 172})


...

/tmp/ipykernel_6527/2922189501.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",


[coord iter 3] → SUBAGENTE: ask_nutritionist({'client_id': 'C3', 'task': 'Desarrollar un plan nutricional para Jorge enfocado en bajar...'})


/tmp/ipykernel_6527/2922189501.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds") + "Z",



[coord iter 4] respuesta final.

INFORME INTEGRAL — JORGE (C3)
**Informe Integral para Jorge (C3)**

---

**1. Evaluación de Línea Base**

- **IMC:** 32.11 (Categoría: Obesidad)
- **Riesgos:** Actualmente no hay suficientes datos para detectar riesgos específicos. Se recomienda un seguimiento regular para ajustar el plan según el progreso.

**2. Plan de Entrenamiento**

**Objetivo:** Bajar de peso y mejorar la condición general, evitando ejercicios de impacto en las rodillas.

**Rutina Semanal:**

- **Lunes: Piernas y Glúteos**
  - Prensa 45°: 3 series de 12 repeticiones
  - Sentadilla Goblet (si no hay dolor): 3 series de 10 repeticiones
  - Hip Thrust: 3 series de 12 repeticiones

- **Martes: Cardio y Movilidad**
  - Bicicleta Estática: 20 minutos a ritmo moderado
  - Movilidad Articular: 15 minutos

- **Miércoles: Espalda y Core**
  - Remo Polea Baja: 3 series de 12 repeticiones
  - Plancha: 3 series de 20-30 segundos
  - Natación Suave: 20 minutos

- **Jueves: Descanso Activo**
  

/tmp/ipykernel_6527/3584532999.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f.write(f"**Fecha:** {datetime.utcnow().isoformat(timespec='seconds')}Z\n\n")


# 11. Fine-tuning supervisado del Entrenador

## Flujo del sistema

```
ARQUETIPOS DE CLIENTE (4)
   │   post_cirugia · hipertrofia_hombre · obesidad_rodilla · hipertrofia_fem_bajo_peso
   ▼ (gpt-4o-mini como profesor, temperature=0.9)
Pares sintéticos (user, assistant)  ~30 por arquetipo → ~120 totales
   │
   ▼ (validación de schema OpenAI chat)
Ejemplos en formato {messages:[system, user, assistant]}
   │
   ▼ (shuffle seed=42 · 85/15)
trainer_ft_train.jsonl   trainer_ft_val.jsonl
   │                         │
   └──────────┬──────────────┘
              ▼ (files.create purpose="fine-tune")
       Upload a OpenAI
              │
              ▼ (fine_tuning.jobs.create · 3 epochs)
       Modelo base: gpt-4o-mini-2024-07-18
              │
              ▼ (~15 min, checkpoint auto en validación)
       ft:gpt-4o-mini:...:fitcoach-trainer:xxxxx
              │
              ▼ (swap en ask_trainer)
       Entrenador destilado, más barato y con estilo FitCoach
```

## Qué estamos destilando

El modelo base (`gpt-4o-mini`) aprende de los ejemplos a:

- Responder con formato fijo: **Objetivo / Rutina / Advertencias**.
- Respetar **reglas duras por arquetipo**:
  - post-cirugía → sin HIIT, sin abdominales directos las primeras 6 semanas.
  - obesidad + rodilla → **cero impacto**, prioridad bicicleta/elíptica/natación.
  - hipertrofia hombre → split push/pull/legs, progresión de carga.
  - hipertrofia femenino bajo peso → volumen moderado-alto, foco en kcal.
- Mantener respuestas **breves** (≤120 palabras), en español profesional.

## Por qué fine-tuning (y no solo prompts más largos)

- **Latencia**: el system prompt del Entrenador deja de necesitar todas las
  reglas explícitas → cada inferencia usa ~60 % menos tokens de contexto.
- **Coste**: `gpt-4o-mini` fine-tuned cuesta una fracción de `gpt-4o` base
  conservando la calidad en este nicho.
- **Consistencia**: el formato `Objetivo / Rutina / Advertencias` sale estable
  incluso con temperaturas altas.
- **Rúbrica**: cubre el requisito explícito de demostrar fine-tuning con
  pipeline reproducible y datos trazables.

## Seguridad del proceso

- **Datos sintéticos etiquetados**, no datos reales de pacientes → sin riesgo
  de fuga de información personal (requisito del bloque ético del proyecto).
- Dataset y split **deterministas** (semilla fija) → reproducible al pie de la
  letra por cualquier evaluador.
- Job lanzado solo si `EJECUTAR_FT=True` → no se incurre en costos
  involuntarios; la ejecución es una decisión explícita.

## Resultado esperado

Tras un FT exitoso, la llamada:

```python
ask_trainer("C3", "rutina para Jorge con problemas de rodilla")
```

usa el modelo `ft:...:fitcoach-trainer:xxxxx` en lugar de `gpt-4o`, y devuelve
la misma calidad (o mejor) con menor coste y latencia. El resto del sistema
(Coordinador, Nutricionista, Analista, memoria, MCP, RAG) permanece intacto.

In [44]:
# === Generar dataset sintético para FT ===
import json, os, time
from tqdm.auto import tqdm

DATASET_DIR = "/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/ft_data"
os.makedirs(DATASET_DIR, exist_ok=True)

ARQUETIPOS = [
    {
        "nombre": "post_cirugia",
        "descripcion": "Mujer 20-40 años, post-operatorio (lipo, abdominoplastia). Objetivo: mantener resultados sin comprometer cicatrización.",
        "reglas": "NO hiit, NO abdominales directos las primeras 6 semanas, NO cargas altas, progresión muy suave.",
    },
    {
        "nombre": "hipertrofia_hombre",
        "descripcion": "Hombre 25-35 años, oficinista, objetivo hipertrofia y definición. 5 días/semana, disciplina baja.",
        "reglas": "Rutinas estructuradas push/pull/legs o torso/pierna, periodización, énfasis en adherencia.",
    },
    {
        "nombre": "obesidad_rodilla",
        "descripcion": "Hombre 50+ años, sobrepeso u obesidad, dolor de rodilla, sedentario previo.",
        "reglas": "NADA de impacto (correr, saltar). Prioridad: bicicleta estática, elíptica, natación, fuerza guiada, movilidad.",
    },
    {
        "nombre": "hipertrofia_femenino_bajo_peso",
        "descripcion": "Mujer 18-22 años, bajo peso, objetivo ganar masa muscular.",
        "reglas": "Volumen moderado-alto, mucha ingesta calórica, énfasis en progresión de carga; evitar sobre-entrenamiento.",
    },
]

SYSTEM_ENTRENADOR_SHORT = (
    "Eres el AGENTE ENTRENADOR de FitCoach, licenciado en educación física. "
    "Respondes en español con formato 'Objetivo:', 'Rutina:', 'Advertencias:'. "
    "Respetas SIEMPRE restricciones médicas. Máx 120 palabras."
)

TEMPLATE_GEN = """Genera {n} pares pregunta-respuesta para entrenar a un modelo con el rol:

ROL DEL ASISTENTE:
Eres el AGENTE ENTRENADOR de FitCoach. Respondes en español con formato fijo
"Objetivo: ...", "Rutina: ...", "Advertencias: ...". Máx ~120 palabras.

ARQUETIPO DE CLIENTE:
{desc}

REGLAS DURAS:
{reglas}

Devuelve una lista JSON válida donde cada item tiene claves "user" y "assistant".
La "user" debe sonar como un cliente real (pregunta natural).
La "assistant" respeta el formato y las reglas duras.

Importante: SOLO la lista JSON pura, sin markdown, sin texto extra."""

def generar_lote(arq: dict, n: int = 10, model="gpt-4o-mini"):
    prompt = TEMPLATE_GEN.format(n=n, desc=arq["descripcion"], reglas=arq["reglas"])
    resp = client_oai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Genera datos de entrenamiento JSON válidos."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.9,
    )
    txt = resp.choices[0].message.content.strip()
    if txt.startswith("```"):
        txt = txt.strip("`")
        if txt.lower().startswith("json"):
            txt = txt[4:].strip()
    try:
        return json.loads(txt)
    except Exception:
        i, j = txt.find("["), txt.rfind("]")
        if i >= 0 and j > i:
            return json.loads(txt[i:j+1])
        raise

all_examples = []
for arq in ARQUETIPOS:
    print(f"Generando arquetipo: {arq['nombre']}")
    for _ in tqdm(range(3), desc=arq["nombre"]):  # 3 lotes × 10 = 30
        try:
            pares = generar_lote(arq, n=10)
            for p in pares:
                all_examples.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_ENTRENADOR_SHORT},
                        {"role": "user", "content": p["user"]},
                        {"role": "assistant", "content": p["assistant"]},
                    ],
                    "_arquetipo": arq["nombre"],
                })
        except Exception as e:
            print(f"  lote falló: {e}")
            time.sleep(2)

print(f"\nTotal de ejemplos generados: {len(all_examples)}")
print("\nEjemplo (primer ítem):")
print(json.dumps(all_examples[0], ensure_ascii=False, indent=2))

Generando arquetipo: post_cirugia


post_cirugia:   0%|          | 0/3 [00:00<?, ?it/s]

Generando arquetipo: hipertrofia_hombre


hipertrofia_hombre:   0%|          | 0/3 [00:00<?, ?it/s]

Generando arquetipo: obesidad_rodilla


obesidad_rodilla:   0%|          | 0/3 [00:00<?, ?it/s]

Generando arquetipo: hipertrofia_femenino_bajo_peso


hipertrofia_femenino_bajo_peso:   0%|          | 0/3 [00:00<?, ?it/s]


Total de ejemplos generados: 120

Ejemplo (primer ítem):
{
  "messages": [
    {
      "role": "system",
      "content": "Eres el AGENTE ENTRENADOR de FitCoach, licenciado en educación física. Respondes en español con formato 'Objetivo:', 'Rutina:', 'Advertencias:'. Respetas SIEMPRE restricciones médicas. Máx 120 palabras."
    },
    {
      "role": "user",
      "content": "¿Qué puedo hacer para mantener mis resultados después de la lipo?"
    },
    {
      "role": "assistant",
      "content": "Objetivo: Mantener resultados post-operatorios. Rutina: Caminatas suaves diarias de 20-30 minutos, estiramientos ligeros. Advertencias: Evitar HIIT y abdominales directos las primeras 6 semanas."
    }
  ],
  "_arquetipo": "post_cirugia"
}


In [45]:
# ===  Validar, split y guardar JSONL ===
import random
from collections import Counter

DATASET_DIR = "/content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/ft_data"

def validar_ejemplo(ex: dict) -> bool:
    if "messages" not in ex: return False
    msgs = ex["messages"]
    if len(msgs) < 2: return False
    roles = [m.get("role") for m in msgs]
    if "user" not in roles or "assistant" not in roles: return False
    for m in msgs:
        if not isinstance(m.get("content"), str) or not m["content"].strip():
            return False
    return True

validos = [ex for ex in all_examples if validar_ejemplo(ex)]
print(f"Ejemplos válidos: {len(validos)} / {len(all_examples)}")

limpios = [{"messages": ex["messages"]} for ex in validos]
random.seed(42)
random.shuffle(limpios)

split = int(len(limpios) * 0.85)
train, val = limpios[:split], limpios[split:]

train_path = os.path.join(DATASET_DIR, "trainer_ft_train.jsonl")
val_path   = os.path.join(DATASET_DIR, "trainer_ft_val.jsonl")

with open(train_path, "w", encoding="utf-8") as f:
    for ex in train:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
with open(val_path, "w", encoding="utf-8") as f:
    for ex in val:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Train → {train_path} ({len(train)} ejemplos)")
print(f"Val   → {val_path} ({len(val)} ejemplos)")

dist = Counter([ex["_arquetipo"] for ex in validos])
print("\nDistribución por arquetipo:")
for k, v in dist.items():
    print(f"  {k}: {v}")

Ejemplos válidos: 110 / 120
Train → /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/ft_data/trainer_ft_train.jsonl (93 ejemplos)
Val   → /content/drive/MyDrive/si7016-JoseBedoya-taller1-261/Trabajo Final/proyecto_entrenador/ft_data/trainer_ft_val.jsonl (17 ejemplos)

Distribución por arquetipo:
  post_cirugia: 30
  hipertrofia_hombre: 30
  obesidad_rodilla: 30
  hipertrofia_femenino_bajo_peso: 20


In [46]:
# === Upload + FT job (opcional) ===

EJECUTAR_FT = True   # cambia a True si quieres lanzar el fine-tune real

if EJECUTAR_FT:
    with open(train_path, "rb") as f:
        tr_file = client_oai.files.create(file=f, purpose="fine-tune")
    with open(val_path, "rb") as f:
        vl_file = client_oai.files.create(file=f, purpose="fine-tune")
    print(f"Train file_id: {tr_file.id}")
    print(f"Val   file_id: {vl_file.id}")

    job = client_oai.fine_tuning.jobs.create(
        training_file=tr_file.id,
        validation_file=vl_file.id,
        model="gpt-4o-mini-2024-07-18",
        suffix="fitcoach-trainer",
        hyperparameters={"n_epochs": 3},
    )
    print(f"\nFT job creado: {job.id}")
    print(f"Estado inicial: {job.status}")
    FT_JOB_ID = job.id
    print(f"\nPara monitorear:")
    print(f"  client_oai.fine_tuning.jobs.retrieve('{FT_JOB_ID}')")
else:
    FT_JOB_ID = None
    print("FT NO EJECUTADO (EJECUTAR_FT=False).")
    print(f"Datasets listos en: {DATASET_DIR}")
    print("Pipeline documentado. Para lanzar el FT real, cambia EJECUTAR_FT=True.")

Train file_id: file-45g6SphpicojKSww2PfCCq
Val   file_id: file-VLUNqi3bGrEowy2CVtwSL2

FT job creado: ftjob-ax2v1oyiHePvGPwmrWRM5f51
Estado inicial: validating_files

Para monitorear:
  client_oai.fine_tuning.jobs.retrieve('ftjob-ax2v1oyiHePvGPwmrWRM5f51')


In [47]:
# ===  Usar el modelo fine-tuned ===

# Rellena cuando el FT termine:
# FT_MODEL_ID = "ft:gpt-4o-mini-2024-07-18:fitcoach:fitcoach-trainer:xxxxx"
FT_MODEL_ID = None

if FT_MODEL_ID:
    def ask_trainer(client_id: str, task: str) -> str:
        return _ask_specialist(SYSTEM_ENTRENADOR, task, client_id, model=FT_MODEL_ID)

    resp = client_oai.chat.completions.create(
        model=FT_MODEL_ID,
        messages=[
            {"role": "system", "content": SYSTEM_ENTRENADOR_SHORT},
            {"role": "user", "content": "Tengo 55 años, sobrepeso y me duelen las rodillas. ¿Rutina?"},
        ],
    )
    print("Respuesta del modelo fine-tuned:")
    print(resp.choices[0].message.content)
    print(f"\nEntrenador ahora usa: {FT_MODEL_ID}")
else:
    print("FT_MODEL_ID aún no configurado. ask_trainer sigue usando gpt-4o.")
    print("Cuando el FT termine (o si decides lanzarlo), vuelve a ejecutar esta celda con FT_MODEL_ID asignado.")

FT_MODEL_ID aún no configurado. ask_trainer sigue usando gpt-4o.
Cuando el FT termine (o si decides lanzarlo), vuelve a ejecutar esta celda con FT_MODEL_ID asignado.


# 12. Interfaz Gradio con entrada de voz (Whisper)

## Flujo del sistema

```
Usuario (navegador Colab)
   │
   ├── texto escrito ──┐
   │                   │
   └── audio (micro) ──▶ Whisper API ("whisper-1")
                       │   transcribe a texto en español
                       ▼
                 Texto del usuario
                       │
                       ▼ (selección cliente C1..C4 + agente)
              agent_run_with_memory / coordinator_run
                       │
                       ▼ (MCP + RAG + sub-agentes)
                 Respuesta del agente
                       │
                       ▼
              gr.Chatbot (historial visible)
                       │
                       ▼ (persistencia automática)
              memory/C{id}.jsonl
```

## Componentes de la UI

- `gr.Dropdown` — selector de cliente (C1, C2, C3, C4).
- `gr.Radio` — selector de agente (Coordinador, Entrenador, Nutricionista, Analista).
- `gr.Audio(sources=["microphone"])` — captura del micrófono, devuelve archivo wav/webm.
- `gr.Textbox` — entrada de texto alternativa.
- `gr.Chatbot` — historial de conversación estilo ChatGPT.
- `gr.Button("Limpiar memoria")` — borra `memory/C{id}.jsonl` para empezar limpio.

## Por qué Whisper + Gradio

- **Whisper API** (modelo `whisper-1`) está optimizado para español con ruido
  ambiental, lo cual importa en un gimnasio. Evita instalar pesos locales y
  aprovecha la misma API key que el resto del sistema.
- **Gradio** corre dentro de Colab con `share=True` generando una URL pública
  temporal → el profesor puede probar la app desde su navegador sin instalar nada.
- La UI es **stateless por diseño**: la memoria persiste en archivos, no en
  variables de sesión. Si se recarga la página el historial sigue ahí.

## Accesibilidad

La voz como modalidad de entrada es clave en el contexto del gimnasio:
- El cliente puede estar entrenando y consultar al sistema sin escribir.
- Usuarios con dificultades de movilidad (post-cirugía) o personas mayores
  (Jorge, 55 años) se benefician de la entrada hablada.
- Requisito alineado con buenas prácticas de UX en salud y deporte.

In [48]:
# === FIX: Parchear HfFolder ===
import huggingface_hub
import os

if not hasattr(huggingface_hub, "HfFolder"):
    class HfFolder:
        path_token = os.path.expanduser("~/.cache/huggingface/token")
        @classmethod
        def get_token(cls):
            return os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        @classmethod
        def save_token(cls, token: str):
            os.environ["HF_TOKEN"] = token
        @classmethod
        def delete_token(cls):
            os.environ.pop("HF_TOKEN", None)
    huggingface_hub.HfFolder = HfFolder
    print("HfFolder shim instalado.")
else:
    print("HfFolder ya existe en huggingface_hub.")

HfFolder shim instalado.


In [60]:
!pip install -q "gradio==4.44.1" "huggingface_hub<0.32" "websockets==12.0"

In [61]:
# === Helper Whisper ===
import gradio as gr
import os

assert 'client_oai' in dir(), "client_oai no existe. Ejecuta antes la celda que crea el cliente de OpenAI."

def transcribir_audio(audio_path: str) -> str:
    """Transcribe audio a texto con Whisper API (español)."""
    if not audio_path or not os.path.isfile(audio_path):
        return ""
    with open(audio_path, "rb") as f:
        tr = client_oai.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            language="es",
        )
    return tr.text.strip()

print("Gradio:", gr.__version__)
print("Helper transcribir_audio listo.")

Gradio: 5.50.0
Helper transcribir_audio listo.


In [62]:
# === FIX: Monkeypatch a gradio_client.utils para evitar crash en api_info ===
import gradio_client.utils as _gcu

_orig_pub = _gcu.json_schema_to_python_type
_orig_priv = _gcu._json_schema_to_python_type

def _safe_pub(schema, defs=None):
    try:
        return _orig_pub(schema, defs) if defs is not None else _orig_pub(schema)
    except TypeError:
        try:
            return _orig_pub(schema)
        except Exception:
            return "Any"
    except Exception:
        return "Any"

def _safe_priv(schema, defs=None):
    try:
        return _orig_priv(schema, defs)
    except Exception:
        return "Any"

_gcu.json_schema_to_python_type = _safe_pub
_gcu._json_schema_to_python_type = _safe_priv

print("Schema-resolver de gradio_client parcheado. api_info ya no va a crashear.")

Schema-resolver de gradio_client parcheado. api_info ya no va a crashear.


In [ ]:
import gradio as gr
import os
from datetime import datetime

# ===  App Gradio con show_api=False ===

AGENT_MAP = {
    "Coordinador":   ("coordinador",   None),
    "Entrenador":    ("entrenador",    SYSTEM_ENTRENADOR),
    "Nutricionista": ("nutricionista", SYSTEM_NUTRICIONISTA),
    "Analista":      ("analista",      SYSTEM_ANALISTA),
}

CLIENT_LABELS = {
    "C1": "C1 — Mariana (post-lipo, 25F)",
    "C2": "C2 — Andrés (hipertrofia, 29M)",
    "C3": "C3 — Jorge (obesidad + rodilla, 55M)",
    "C4": "C4 — Valentina (hipertrofia bajo peso, 18F)",
}
LABEL_TO_ID = {v: k for k, v in CLIENT_LABELS.items()}
ID_TO_LABEL = CLIENT_LABELS


def cargar_historial(label_cliente: str):
    client_id = LABEL_TO_ID[label_cliente]
    eps = load_recent_episodes(client_id, n=40)
    mensajes = []
    for e in eps:
        if e["role"] == "user":
            mensajes.append({"role": "user", "content": e["content"]})
        elif e["role"] == "assistant":
            mensajes.append({"role": "assistant", "content": e["content"]})
    return mensajes


def responder(label_cliente, agente_label, audio_path, texto, historial):
    historial = historial or []
    client_id = LABEL_TO_ID[label_cliente]

    if audio_path:
        try:
            user_text = transcribir_audio(audio_path)
        except Exception as e:
            return historial, "", None, f"❌ Error transcribiendo: {e}"
    else:
        user_text = (texto or "").strip()

    if not user_text:
        return historial, "", None, "⚠️ Escribe un mensaje o graba audio."

    agent_key, sys_prompt = AGENT_MAP[agente_label]
    try:
        if agent_key == "coordinador":
            out = coordinator_run(user_text, client_id=client_id, verbose=False)
        else:
            out = agent_run_with_memory(
                client_id=client_id,
                system_prompt=sys_prompt,
                user_message=user_text,
                agent_name=agent_key,
                verbose=False,
            )
        respuesta = out["answer"]
        n_tools = len(out.get("trace", []))
        status = f"✅ {agente_label} respondió. Tools usadas: {n_tools}"
    except Exception as e:
        respuesta = f"Error: {type(e).__name__}: {e}"
        status = "❌ Falló la llamada al agente."

    historial.append({"role": "user", "content": user_text})
    historial.append({"role": "assistant", "content": respuesta})
    return historial, "", None, status


def limpiar_mem(label_cliente):
    client_id = LABEL_TO_ID[label_cliente]
    clear_memory(client_id)
    return [], f"🧹 Memoria de {client_id} borrada."


def al_cambiar_cliente(label_cliente):
    client_id = LABEL_TO_ID[label_cliente]
    return cargar_historial(label_cliente), f"Historial cargado para {client_id}."


# --- UI ---
with gr.Blocks(title="FitCoach Multi-Agente", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🏋️ FitCoach — Sistema Multi-Agente de Entrenamiento Personal")
    gr.Markdown("Proyecto final SI7016 · Universidad EAFIT 2026-1 · MCP + RAG + GPT-4 + Whisper")

    with gr.Row():
        cliente = gr.Dropdown(
            choices=list(CLIENT_LABELS.values()),
            value=CLIENT_LABELS["C3"],
            label="Cliente",
            scale=2,
        )
        agente = gr.Radio(
            choices=list(AGENT_MAP.keys()),
            value="Coordinador",
            label="Agente",
            scale=3,
        )

    chat = gr.Chatbot(label="Conversación", height=420, type="messages")

    with gr.Row():
        texto = gr.Textbox(placeholder="Escribe tu mensaje…", scale=4,
                           label="Texto", lines=2)
        audio = gr.Audio(sources=["microphone"], type="filepath",
                         label="🎤 Voz (Whisper)", scale=2)

    with gr.Row():
        btn_enviar = gr.Button("Enviar", variant="primary", scale=3)
        btn_limpiar = gr.Button("🧹 Limpiar memoria", scale=1)

    status = gr.Markdown("")

    cliente.change(al_cambiar_cliente, inputs=cliente, outputs=[chat, status])
    btn_enviar.click(responder,
                     inputs=[cliente, agente, audio, texto, chat],
                     outputs=[chat, texto, audio, status])
    texto.submit(responder,
                 inputs=[cliente, agente, audio, texto, chat],
                 outputs=[chat, texto, audio, status])
    btn_limpiar.click(limpiar_mem, inputs=cliente, outputs=[chat, status])
    app.load(al_cambiar_cliente, inputs=cliente, outputs=[chat, status])

app.launch(share=True, debug=False, show_api=False)

/tmp/ipykernel_6527/2382004945.py:87: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="FitCoach Multi-Agente", theme=gr.themes.Soft()) as app:
/tmp/ipykernel_6527/2382004945.py:105: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat = gr.Chatbot(label="Conversación", height=420, type="messages")
/tmp/ipykernel_6527/2382004945.py:129: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  app.launch(share=True, debug=False, show_api=False)
Exception in thread Thread-121 (run):
Traceback (most recent call last):
  Fil